## 1) ربط Google Drive (لحفظ النتائج بشكل دائم)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/heliyon_6g"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "sixg"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "results"), exist_ok=True)
%cd $PROJECT_DIR
print("working directory:", os.getcwd())


Mounted at /content/drive
/content/drive/MyDrive/heliyon_6g
working directory: /content/drive/MyDrive/heliyon_6g


## 2) تثبيت المتطلبات

In [ ]:
%%writefile requirements.txt
numpy>=1.24
scipy>=1.10
pandas>=2.0
matplotlib>=3.7
torch>=2.0


Writing requirements.txt


In [ ]:
!pip install -q -r requirements.txt


## 3) كتابة حزمة الكود `sixg/`

In [ ]:
%%writefile sixg/__init__.py


Writing sixg/__init__.py


In [ ]:
%%writefile sixg/env.py
"""
6G edge downlink environment with deadline-constrained queueing.

Design notes that matter for the review response
------------------------------------------------
1.  TRUE MDP, not a bandit.  Each user holds a FIFO packet queue with per-packet
    deadlines; the allocation at slot t drains queues and therefore determines
    the state at t+1.

2.  PACKET-LEVEL RELIABILITY.  PDR is the fraction of ARRIVED packets that
    expire before being fully served (plus buffer overflow).  It is a
    deadline-violation rate, which is what the paper should claim -- not
    "3GPP URLLC 1e-5", which this system does not and cannot demonstrate.

3.  GOODPUT, NOT SHANNON RATE.  A user with an empty queue cannot use the
    bandwidth assigned to it.  The delivered rate is min(R_i, backlog_i/TTI),
    so over-provisioning a strong-but-idle user is correctly penalised.  This
    is why a pure Greedy/SINR-maximising allocation does badly here.

4.  ACTUAL POWER CONTROL.  The action carries a total-power scaling factor
    rho in [RHO_MIN, 1], so the agent can spend less than P_max.  Without it,
    "energy efficiency" is just a rescaled throughput and any Green claim is
    vacuous.

Queues are stored as an N x (D+1) count matrix Q[i, k] = number of packets of
user i with k slots of life remaining, which makes every operation vectorised.
"""

import numpy as np

CFG = dict(
    # --- radio ---
    W_TOTAL_HZ      = 200e6,
    P_MAX_DBM       = 46.0,
    NOISE_PSD_DBM   = -174.0,
    NF_DB           = 7.0,
    P_CIRC_W        = 10.0,
    XI_PA           = 1.2,
    D_MIN_M         = 50.0,
    D_MAX_M         = 500.0,
    SHADOW_STD_DB   = 8.0,
    RHO_FADING      = 0.90,
    INTERF_DBM      = -95.0,
    RHO_MIN         = 0.05,

    # --- traffic / queueing ---
    TTI_S           = 1e-3,
    PKT_BITS        = 8000.0,
    DEADLINE_SLOTS  = 5,
    LAMBDA_MIN_MBPS = 5.0,
    LAMBDA_MAX_MBPS = 25.0,
    QUEUE_CAP_PKTS  = 200,

    # --- episode ---
    T_SLOTS         = 40,

    # --- objective weights (w1..w4 of Eq. 8) ---
    W_THR           = 0.35,
    W_FAIR          = 0.15,
    W_EE            = 0.20,
    W_PDR           = 0.60,

    MIN_FRAC        = 1e-4,
)


def dbm_to_w(x):
    return 10.0 ** ((x - 30.0) / 10.0)


class SixGEdgeEnv:
    def __init__(self, n_users=30, cfg=None, seed=0):
        self.cfg = dict(CFG) if cfg is None else dict(cfg)
        self.N = int(n_users)
        self.rng = np.random.default_rng(seed)
        c = self.cfg
        self.W = c["W_TOTAL_HZ"]
        self.P_max = dbm_to_w(c["P_MAX_DBM"])
        self.N0 = dbm_to_w(c["NOISE_PSD_DBM"] + c["NF_DB"])
        self.I_inter = dbm_to_w(c["INTERF_DBM"])
        self.D = int(c["DEADLINE_SLOTS"])
        self.T = c["T_SLOTS"]
        self.state_dim = 4 * self.N
        self.action_dim = 2 * self.N + 1

    # ---------------------------------------------------------------- reset
    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        c, N = self.cfg, self.N

        self.dist_m = self.rng.uniform(c["D_MIN_M"], c["D_MAX_M"], N)
        pl_db = 128.1 + 37.6 * np.log10(self.dist_m / 1000.0)
        shadow_db = self.rng.normal(0.0, c["SHADOW_STD_DB"], N)
        self.large_scale = 10.0 ** ((-pl_db + shadow_db) / 10.0)
        self.h = (self.rng.normal(0, np.sqrt(0.5), N)
                  + 1j * self.rng.normal(0, np.sqrt(0.5), N))

        self.lam_mbps = self.rng.uniform(c["LAMBDA_MIN_MBPS"], c["LAMBDA_MAX_MBPS"], N)
        self.lam_pkts = self.lam_mbps * 1e6 * c["TTI_S"] / c["PKT_BITS"]

        self.R_ref = float(self.lam_mbps.sum())
        self.EE_ref = self.R_ref / (c["P_CIRC_W"] + c["XI_PA"] * self.P_max)

        self.Q = np.zeros((N, self.D + 1))
        self.residual = np.zeros(N)
        self.avg_rate = np.full(N, 1.0)         # EMA of delivered rate (PF anchor)

        self.t = 0
        self.tot_arrived = 0
        self.tot_dropped = 0
        self.tot_delivered_bits = 0.0
        return self._observe()

    # ------------------------------------------------------------ observation
    def _gain(self):
        return self.large_scale * (np.abs(self.h) ** 2)

    def _queue_pkts(self):
        return self.Q.sum(axis=1)

    def _observe(self):
        c, N = self.cfg, self.N
        g = self._gain()
        snr_ref = (self.P_max / N) * g / (self.N0 * self.W / N + self.I_inter)
        s_snr = np.log10(snr_ref + 1e-12) / 3.0
        s_q = np.minimum(self._queue_pkts() / 50.0, 4.0)
        occupied = self.Q > 0
        hol = np.where(occupied.any(axis=1),
                       np.argmax(occupied, axis=1), self.D).astype(float)
        s_hol = hol / self.D
        s_lam = self.lam_mbps / c["LAMBDA_MAX_MBPS"]
        return np.concatenate([s_snr, s_q, s_hol, s_lam]).astype(np.float64)

    # --------------------------------------------------------------- physics
    def _rates_core(self, B, P, RHO):
        c = self.cfg
        B = np.maximum(B, c["MIN_FRAC"])
        P = np.maximum(P, c["MIN_FRAC"])
        g = self._gain()[None, :]
        bw = B * self.W
        noise = self.N0 * bw + self.I_inter
        p_w = P * (RHO[:, None] * self.P_max)
        sinr = p_w * g / noise
        return bw * np.log2(1.0 + sinr) / 1e6

    def evaluate_batch(self, B, P, RHO):
        c = self.cfg
        R = self._rates_core(B, P, RHO)
        backlog_bits = np.maximum(self._queue_pkts() - self.residual, 0.0) * c["PKT_BITS"]
        cap_mbps = backlog_bits / c["TTI_S"] / 1e6
        U = np.minimum(R, cap_mbps[None, :])
        thr = U.sum(axis=1)
        # Demand-aware fairness: Jain's index over the per-user satisfaction
        # ratio (delivered rate / offered load).  Plain Jain over raw rates
        # would penalise a scheduler for correctly giving less to users who
        # asked for less.
        sat = U / (self.lam_mbps[None, :] + 1e-9)
        jain = (sat.sum(axis=1) ** 2) / (self.N * np.sum(sat ** 2, axis=1) + 1e-12)
        p_tot = c["P_CIRC_W"] + c["XI_PA"] * RHO * self.P_max
        ee = thr / p_tot

        urgent_bits = self.Q[:, 1] * c["PKT_BITS"]
        deliver = R * 1e6 * c["TTI_S"]
        miss = np.maximum(urgent_bits[None, :] - deliver, 0.0).sum(axis=1)
        pdr_proxy = miss / (urgent_bits.sum() + 1e-9)

        f = (c["W_THR"] * thr / self.R_ref
             + c["W_FAIR"] * jain
             + c["W_EE"] * ee / self.EE_ref
             - c["W_PDR"] * pdr_proxy)
        return f, dict(thr=thr, jain=jain, ee=ee, pdr_proxy=pdr_proxy, R=R, U=U)

    def evaluate(self, b, p, rho=1.0):
        f, d = self.evaluate_batch(np.atleast_2d(b), np.atleast_2d(p),
                                   np.atleast_1d(rho))
        return float(f[0]), {k: v[0] for k, v in d.items()}

    # -------------------------------------------------------------- dynamics
    def step(self, b, p, rho=1.0):
        c, N = self.cfg, self.N
        f, info = self.evaluate(b, p, rho)
        R = np.atleast_1d(info["R"])

        avail_pkts = R * 1e6 * c["TTI_S"] / c["PKT_BITS"] + self.residual
        q_tot = self._queue_pkts()
        n_full = np.floor(avail_pkts)
        served = np.minimum(n_full, q_tot)
        self.residual = np.where(n_full <= q_tot, avail_pkts - n_full, 0.0)

        cum = np.cumsum(self.Q, axis=1)
        remove = np.clip(served[:, None] - (cum - self.Q), 0.0, self.Q)
        self.Q = self.Q - remove
        self.tot_delivered_bits += float(served.sum() * c["PKT_BITS"])
        served_mbps = served * c["PKT_BITS"] / c["TTI_S"] / 1e6
        self.avg_rate = 0.9 * self.avg_rate + 0.1 * served_mbps

        expired = float(self.Q[:, 1].sum())
        self.Q[:, 1] = 0.0
        self.Q[:, :-1] = self.Q[:, 1:]
        self.Q[:, -1] = 0.0
        self.tot_dropped += int(round(expired))

        arrivals = self.rng.poisson(self.lam_pkts).astype(float)
        free = np.maximum(c["QUEUE_CAP_PKTS"] - self._queue_pkts(), 0.0)
        admit = np.minimum(arrivals, free)
        self.tot_dropped += int(round(float((arrivals - admit).sum())))
        self.Q[:, self.D] += admit
        self.tot_arrived += int(arrivals.sum())

        rho_f = c["RHO_FADING"]
        innov = (self.rng.normal(0, np.sqrt(0.5), N)
                 + 1j * self.rng.normal(0, np.sqrt(0.5), N))
        self.h = rho_f * self.h + np.sqrt(1.0 - rho_f ** 2) * innov

        self.t += 1
        return self._observe(), float(f), self.t >= self.T, info

    def episode_stats(self):
        c = self.cfg
        return dict(
            pdr=100.0 * self.tot_dropped / max(self.tot_arrived, 1),
            goodput=self.tot_delivered_bits / (self.t * c["TTI_S"]) / 1e6,
            offered=float(self.lam_mbps.sum()),
        )


# ----------------------------------------------------------------------------
def project_action(x, n):
    x = np.asarray(x, dtype=float)
    xb = np.abs(x[:n]) + 1e-9
    xp = np.abs(x[n:2 * n]) + 1e-9
    b = xb / xb.sum()
    p = xp / xp.sum()
    r = 0.5 * (np.tanh(x[2 * n]) + 1.0) if x.shape[0] > 2 * n else 1.0
    rho = CFG["RHO_MIN"] + (1.0 - CFG["RHO_MIN"]) * r
    return b, p, float(rho)


def project_batch(X, n):
    X = np.atleast_2d(X)
    B = np.abs(X[:, :n]) + 1e-9
    P = np.abs(X[:, n:2 * n]) + 1e-9
    B = B / B.sum(axis=1, keepdims=True)
    P = P / P.sum(axis=1, keepdims=True)
    r = 0.5 * (np.tanh(X[:, 2 * n]) + 1.0)
    RHO = CFG["RHO_MIN"] + (1.0 - CFG["RHO_MIN"]) * r
    return B, P, RHO


def inverse_project(b, p, rho):
    r = (rho - CFG["RHO_MIN"]) / (1.0 - CFG["RHO_MIN"])
    r = np.clip(2.0 * r - 1.0, -0.999, 0.999)
    return np.concatenate([b / (b.max() + 1e-12),
                           p / (p.max() + 1e-12),
                           [np.arctanh(r)]])


Writing sixg/env.py


In [ ]:
%%writefile sixg/heuristics.py
"""
Deterministic heuristic anchors and an elitist, vectorised Grey Wolf Optimizer.

GWO follows Mirjalili, Mirjalili & Lewis, "Grey Wolf Optimizer", Advances in
Engineering Software 69:46-61, 2014:
    a decreases linearly from 2 to 0
    A = 2*a*r1 - a,  C = 2*r2
    D_k = |C_k X_k - X|,  X_k' = X_k - A_k D_k   for k in {alpha, beta, delta}
    X(t+1) = mean(X_alpha', X_beta', X_delta')
Positions are projected onto the feasible set after every update, and the
returned solution is the arg-max over every point ever evaluated (elitism).

Each candidate is a vector in R^{2N+1} -> (b, p, rho) via project_batch.
"""

import numpy as np
from .env import project_batch, inverse_project

# Classical heuristics operate at full transmit power, as in their standard
# formulations; the learned policies may scale total power via rho.
FULL_POWER = 1.0


def h_round_robin(env):
    n = env.N
    return np.ones(n) / n, np.ones(n) / n, FULL_POWER


def h_greedy(env):
    g = env._gain()
    w = g / (g.sum() + 1e-30)
    return w.copy(), w.copy(), FULL_POWER


def h_water_filling(env, iters=40):
    """Equal bandwidth split, power by water-filling (bisection on the level)."""
    n = env.N
    b = np.ones(n) / n
    g = env._gain()
    noise = env.N0 * (b * env.W) + env.I_inter
    inv_g = noise / (g + 1e-30)
    lo, hi = float(inv_g.min()), float(inv_g.max() + env.P_max)
    for _ in range(iters):
        mu = 0.5 * (lo + hi)
        if np.maximum(mu - inv_g, 0.0).sum() > env.P_max:
            hi = mu
        else:
            lo = mu
    p = np.maximum(0.5 * (lo + hi) - inv_g, 0.0)
    if p.sum() <= 1e-12:
        p = np.ones(n)
    return b, p / p.sum(), FULL_POWER


def h_urgency(env):
    """Backlog- and deadline-aware anchor (part of the proposed EHS ensemble).

    Not a classical heuristic: it is introduced here, and the ablation reports
    its contribution separately so the gain remains attributable.
    """
    c, n = env.cfg, env.N
    q = env._queue_pkts()
    occ = env.Q > 0
    hol = np.where(occ.any(axis=1), np.argmax(occ, axis=1), env.D).astype(float)
    urgency = (q + 0.5) * (env.D + 1.0 - hol)
    g = env._gain()
    score = urgency * np.sqrt(g / (g.mean() + 1e-30))
    if score.sum() <= 0:
        score = np.ones(n)
    w = score / score.sum()
    return w.copy(), w.copy(), FULL_POWER


def h_prop_fair(env):
    """Proportional-fair anchor: weight by instantaneous rate over the EMA of
    the delivered rate (Kelly / PF scheduling)."""
    n = env.N
    g = env._gain()
    inst = np.log2(1.0 + (env.P_max / n) * g / (env.N0 * env.W / n + env.I_inter))
    score = inst / (env.avg_rate + 1e-3)
    if score.sum() <= 0:
        score = np.ones(n)
    w = score / score.sum()
    return w.copy(), w.copy(), FULL_POWER


ANCHORS = {"WF": h_water_filling, "RR": h_round_robin, "GRD": h_greedy,
           "URG": h_urgency, "PF": h_prop_fair}
CLASSICAL = ("WF", "RR", "GRD", "PF")
ENSEMBLE = ("WF", "RR", "GRD", "URG", "PF")


def seeds_to_matrix(seeds):
    return np.array([inverse_project(b, p, r) for (b, p, r) in seeds])


def gwo_refine(env, seeds, pop_size=10, iters=10, rng=None):
    """Elitist GWO.

    Returns ((b, p, rho), fitness, n_fitness_evals).

    Elitism guarantee: the returned point maximises F over all evaluated
    points, the initial seeds included.  Therefore  F* >= max_k F(seed_k)
    holds by construction for the same F used throughout -- this is what
    Proposition 1 in the manuscript should state, and nothing stronger.
    """
    rng = np.random.default_rng() if rng is None else rng
    n = env.N
    X = seeds_to_matrix(seeds)

    if len(X) < pop_size:
        base = X[0]
        extra = np.abs(base[None, :] + rng.normal(0.0, 0.10, (pop_size - len(X), 2 * n + 1)))
        extra[:, -1] = base[-1] + rng.normal(0.0, 0.3, pop_size - len(X))
        X = np.vstack([X, extra])
    X = X[:pop_size]

    n_evals = 0

    def fitness(M):
        nonlocal n_evals
        B, P, RHO = project_batch(M, n)
        n_evals += M.shape[0]
        return env.evaluate_batch(B, P, RHO)[0]

    scores = fitness(X)
    order = np.argsort(-scores)
    best_vec, best_f = X[order[0]].copy(), float(scores[order[0]])
    Xa, Xb, Xd = X[order[0]].copy(), X[order[1]].copy(), X[order[2]].copy()

    for it in range(iters):
        a = 2.0 - 2.0 * it / max(iters, 1)
        newX = np.zeros_like(X)
        for lead in (Xa, Xb, Xd):
            r1 = rng.random(X.shape)
            r2 = rng.random(X.shape)
            A = 2.0 * a * r1 - a
            C = 2.0 * r2
            Dm = np.abs(C * lead[None, :] - X)
            newX += lead[None, :] - A * Dm
        X = newX / 3.0

        scores = fitness(X)
        order = np.argsort(-scores)
        if scores[order[0]] > best_f:
            best_f = float(scores[order[0]])
            best_vec = X[order[0]].copy()
        Xa, Xb, Xd = X[order[0]].copy(), X[order[1]].copy(), X[order[2]].copy()

    B, P, RHO = project_batch(best_vec[None, :], n)
    return (B[0], P[0], float(RHO[0])), best_f, n_evals


Writing sixg/heuristics.py


In [ ]:
%%writefile sixg/agents.py
"""
Reference implementations of SAC, TD3 and PPO for continuous control.

All three share the same MLP capacity, the same optimiser family, the same
batch size and the same number of environment interactions, so that the
comparison is not confounded by network size or training budget.

SAC  : Haarnoja et al., ICML 2018 (twin critics, tanh-Gaussian policy,
       automatic temperature tuning with target entropy -dim(A)).
TD3  : Fujimoto et al., ICML 2018 (twin critics, delayed policy updates,
       target policy smoothing).
PPO  : Schulman et al., 2017 (clipped surrogate objective, GAE-lambda).
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEV = torch.device("cpu")
LOG_STD_MIN, LOG_STD_MAX = -5.0, 2.0

HP = dict(
    hidden        = 128,
    lr_actor      = 3e-4,
    lr_critic     = 3e-4,
    lr_alpha      = 3e-4,
    gamma         = 0.95,
    tau           = 0.005,
    batch_size    = 128,
    buffer_size   = 100_000,
    start_steps   = 500,      # uniform-random warm-up before learning
    updates_per_step = 1,
    # TD3
    policy_delay  = 2,
    expl_noise    = 0.1,
    target_noise  = 0.2,
    noise_clip    = 0.5,
    # PPO
    ppo_epochs    = 10,
    ppo_clip      = 0.2,
    ppo_lam       = 0.95,
    ppo_rollout   = 400,
    ppo_minibatch = 128,
    # imitation (proposed method only)
    bc_coef_init  = 1.0,
    bc_decay      = 0.995,
)


def mlp(inp, out, hidden, out_act=None):
    layers = [nn.Linear(inp, hidden), nn.ReLU(),
              nn.Linear(hidden, hidden), nn.ReLU(),
              nn.Linear(hidden, out)]
    if out_act is not None:
        layers.append(out_act)
    return nn.Sequential(*layers)


class ReplayBuffer:
    def __init__(self, sdim, adim, size):
        self.s = np.zeros((size, sdim), dtype=np.float32)
        self.a = np.zeros((size, adim), dtype=np.float32)
        self.r = np.zeros((size, 1), dtype=np.float32)
        self.s2 = np.zeros((size, sdim), dtype=np.float32)
        self.d = np.zeros((size, 1), dtype=np.float32)
        self.ptr, self.n, self.size = 0, 0, size

    def add(self, s, a, r, s2, d):
        i = self.ptr
        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, d
        self.ptr = (self.ptr + 1) % self.size
        self.n = min(self.n + 1, self.size)

    def sample(self, bs):
        idx = np.random.randint(0, self.n, bs)
        t = lambda x: torch.as_tensor(x[idx], device=DEV)
        return t(self.s), t(self.a), t(self.r), t(self.s2), t(self.d)


# ------------------------------------------------------------------------ SAC
class SACActor(nn.Module):
    def __init__(self, sdim, adim, hidden):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(sdim, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU())
        self.mu = nn.Linear(hidden, adim)
        self.log_std = nn.Linear(hidden, adim)

    def forward(self, s, deterministic=False, with_logp=True):
        h = self.body(s)
        mu = self.mu(h)
        log_std = torch.clamp(self.log_std(h), LOG_STD_MIN, LOG_STD_MAX)
        std = log_std.exp()
        if deterministic:
            u = mu
        else:
            u = mu + std * torch.randn_like(mu)
        a = torch.tanh(u)
        logp = None
        if with_logp and not deterministic:
            logp = (-0.5 * ((u - mu) / std) ** 2 - log_std
                    - 0.5 * np.log(2 * np.pi)).sum(-1, keepdim=True)
            logp = logp - (2 * (np.log(2) - u - F.softplus(-2 * u))).sum(-1, keepdim=True)
        return a, logp, torch.tanh(mu)


class SAC:
    name = "SAC"

    def __init__(self, sdim, adim, hp=None, seed=0):
        self.hp = dict(HP) if hp is None else dict(hp)
        torch.manual_seed(seed)
        h = self.hp["hidden"]
        self.actor = SACActor(sdim, adim, h).to(DEV)
        self.q1 = mlp(sdim + adim, 1, h).to(DEV)
        self.q2 = mlp(sdim + adim, 1, h).to(DEV)
        self.q1t = mlp(sdim + adim, 1, h).to(DEV)
        self.q2t = mlp(sdim + adim, 1, h).to(DEV)
        self.q1t.load_state_dict(self.q1.state_dict())
        self.q2t.load_state_dict(self.q2.state_dict())

        self.opt_a = torch.optim.Adam(self.actor.parameters(), lr=self.hp["lr_actor"])
        self.opt_q = torch.optim.Adam(list(self.q1.parameters()) + list(self.q2.parameters()),
                                      lr=self.hp["lr_critic"])
        self.log_alpha = torch.zeros(1, requires_grad=True, device=DEV)
        self.opt_alpha = torch.optim.Adam([self.log_alpha], lr=self.hp["lr_alpha"])
        self.target_entropy = -float(adim)

        self.buf = ReplayBuffer(sdim, adim, self.hp["buffer_size"])
        self.adim = adim
        self.steps = 0
        self.bc_coef = self.hp["bc_coef_init"]

    @torch.no_grad()
    def act(self, s, deterministic=False):
        st = torch.as_tensor(s, dtype=torch.float32, device=DEV).unsqueeze(0)
        a, _, mu = self.actor(st, deterministic=deterministic, with_logp=False)
        return (mu if deterministic else a).cpu().numpy()[0]

    def store(self, *tr):
        self.buf.add(*tr)

    def update(self, bc_target=None):
        hp = self.hp
        if self.buf.n < max(hp["batch_size"], hp["start_steps"]):
            return
        s, a, r, s2, d = self.buf.sample(hp["batch_size"])
        alpha = self.log_alpha.exp().detach()

        with torch.no_grad():
            a2, logp2, _ = self.actor(s2)
            qt = torch.min(self.q1t(torch.cat([s2, a2], -1)),
                           self.q2t(torch.cat([s2, a2], -1)))
            target = r + hp["gamma"] * (1 - d) * (qt - alpha * logp2)

        sa = torch.cat([s, a], -1)
        loss_q = F.mse_loss(self.q1(sa), target) + F.mse_loss(self.q2(sa), target)
        self.opt_q.zero_grad(); loss_q.backward(); self.opt_q.step()

        anew, logp, mu = self.actor(s)
        q = torch.min(self.q1(torch.cat([s, anew], -1)),
                      self.q2(torch.cat([s, anew], -1)))
        loss_a = (alpha * logp - q).mean()

        # optional behaviour-cloning term toward the GWO-refined expert action
        if bc_target is not None and self.bc_coef > 1e-3:
            bt = torch.as_tensor(bc_target, dtype=torch.float32, device=DEV).unsqueeze(0)
            st = s[:1]
            _, _, mu_s = self.actor(st, with_logp=False)
            loss_a = loss_a + self.bc_coef * F.mse_loss(mu_s, bt)

        self.opt_a.zero_grad(); loss_a.backward(); self.opt_a.step()

        loss_alpha = -(self.log_alpha.exp() * (logp.detach() + self.target_entropy)).mean()
        self.opt_alpha.zero_grad(); loss_alpha.backward(); self.opt_alpha.step()

        with torch.no_grad():
            for net, tgt in ((self.q1, self.q1t), (self.q2, self.q2t)):
                for p, pt in zip(net.parameters(), tgt.parameters()):
                    pt.mul_(1 - hp["tau"]).add_(hp["tau"] * p)
        self.bc_coef *= hp["bc_decay"]


# ------------------------------------------------------------------------ TD3
class TD3:
    name = "TD3"

    def __init__(self, sdim, adim, hp=None, seed=0):
        self.hp = dict(HP) if hp is None else dict(hp)
        torch.manual_seed(seed)
        h = self.hp["hidden"]
        self.actor = mlp(sdim, adim, h, nn.Tanh()).to(DEV)
        self.actor_t = mlp(sdim, adim, h, nn.Tanh()).to(DEV)
        self.actor_t.load_state_dict(self.actor.state_dict())
        self.q1 = mlp(sdim + adim, 1, h).to(DEV)
        self.q2 = mlp(sdim + adim, 1, h).to(DEV)
        self.q1t = mlp(sdim + adim, 1, h).to(DEV)
        self.q2t = mlp(sdim + adim, 1, h).to(DEV)
        self.q1t.load_state_dict(self.q1.state_dict())
        self.q2t.load_state_dict(self.q2.state_dict())
        self.opt_a = torch.optim.Adam(self.actor.parameters(), lr=self.hp["lr_actor"])
        self.opt_q = torch.optim.Adam(list(self.q1.parameters()) + list(self.q2.parameters()),
                                      lr=self.hp["lr_critic"])
        self.buf = ReplayBuffer(sdim, adim, self.hp["buffer_size"])
        self.adim = adim
        self.it = 0

    @torch.no_grad()
    def act(self, s, deterministic=False):
        st = torch.as_tensor(s, dtype=torch.float32, device=DEV).unsqueeze(0)
        a = self.actor(st).cpu().numpy()[0]
        if not deterministic:
            a = np.clip(a + np.random.normal(0, self.hp["expl_noise"], self.adim), -1, 1)
        return a

    def store(self, *tr):
        self.buf.add(*tr)

    def update(self, bc_target=None):
        hp = self.hp
        if self.buf.n < max(hp["batch_size"], hp["start_steps"]):
            return
        self.it += 1
        s, a, r, s2, d = self.buf.sample(hp["batch_size"])
        with torch.no_grad():
            noise = (torch.randn_like(a) * hp["target_noise"]
                     ).clamp(-hp["noise_clip"], hp["noise_clip"])
            a2 = (self.actor_t(s2) + noise).clamp(-1, 1)
            qt = torch.min(self.q1t(torch.cat([s2, a2], -1)),
                           self.q2t(torch.cat([s2, a2], -1)))
            target = r + hp["gamma"] * (1 - d) * qt
        sa = torch.cat([s, a], -1)
        loss_q = F.mse_loss(self.q1(sa), target) + F.mse_loss(self.q2(sa), target)
        self.opt_q.zero_grad(); loss_q.backward(); self.opt_q.step()

        if self.it % hp["policy_delay"] == 0:
            loss_a = -self.q1(torch.cat([s, self.actor(s)], -1)).mean()
            self.opt_a.zero_grad(); loss_a.backward(); self.opt_a.step()
            with torch.no_grad():
                for net, tgt in ((self.q1, self.q1t), (self.q2, self.q2t),
                                 (self.actor, self.actor_t)):
                    for p, pt in zip(net.parameters(), tgt.parameters()):
                        pt.mul_(1 - hp["tau"]).add_(hp["tau"] * p)


# ------------------------------------------------------------------------ PPO
class PPO:
    name = "PPO"

    def __init__(self, sdim, adim, hp=None, seed=0):
        self.hp = dict(HP) if hp is None else dict(hp)
        torch.manual_seed(seed)
        h = self.hp["hidden"]
        self.pi = mlp(sdim, adim, h, nn.Tanh()).to(DEV)
        self.log_std = nn.Parameter(torch.full((adim,), -0.5, device=DEV))
        self.v = mlp(sdim, 1, h).to(DEV)
        self.opt = torch.optim.Adam(
            list(self.pi.parameters()) + [self.log_std] + list(self.v.parameters()),
            lr=self.hp["lr_actor"])
        self.reset_rollout()
        self.adim = adim

    def reset_rollout(self):
        self.S, self.A, self.R, self.D, self.LP, self.V = [], [], [], [], [], []

    @torch.no_grad()
    def act(self, s, deterministic=False):
        st = torch.as_tensor(s, dtype=torch.float32, device=DEV).unsqueeze(0)
        mu = self.pi(st)
        if deterministic:
            self._last = (st, mu, torch.zeros(1, 1))
            return mu.cpu().numpy()[0]
        std = self.log_std.exp()
        a = mu + std * torch.randn_like(mu)
        logp = (-0.5 * ((a - mu) / std) ** 2 - self.log_std
                - 0.5 * np.log(2 * np.pi)).sum(-1, keepdim=True)
        self._last = (st, a, logp)
        return a.clamp(-1, 1).cpu().numpy()[0]

    def store(self, s, a, r, s2, d):
        st, at, lp = self._last
        self.S.append(st); self.A.append(at); self.LP.append(lp)
        self.R.append(r); self.D.append(d)
        with torch.no_grad():
            self.V.append(self.v(st))

    def update(self, bc_target=None):
        hp = self.hp
        if len(self.R) < hp["ppo_rollout"]:
            return
        S = torch.cat(self.S); A = torch.cat(self.A)
        LP = torch.cat(self.LP).detach(); V = torch.cat(self.V).squeeze(-1)
        R = torch.as_tensor(self.R, dtype=torch.float32, device=DEV)
        D = torch.as_tensor(self.D, dtype=torch.float32, device=DEV)

        adv = torch.zeros_like(R); last = 0.0
        for t in reversed(range(len(R))):
            nextv = V[t + 1] if t + 1 < len(R) else 0.0
            delta = R[t] + hp["gamma"] * nextv * (1 - D[t]) - V[t]
            last = delta + hp["gamma"] * hp["ppo_lam"] * (1 - D[t]) * last
            adv[t] = last
        ret = adv + V
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        idx = np.arange(len(R))
        for _ in range(hp["ppo_epochs"]):
            np.random.shuffle(idx)
            for k in range(0, len(idx), hp["ppo_minibatch"]):
                j = idx[k:k + hp["ppo_minibatch"]]
                mu = self.pi(S[j]); std = self.log_std.exp()
                logp = (-0.5 * ((A[j] - mu) / std) ** 2 - self.log_std
                        - 0.5 * np.log(2 * np.pi)).sum(-1, keepdim=True)
                ratio = (logp - LP[j]).exp().squeeze(-1)
                s1 = ratio * adv[j]
                s2 = ratio.clamp(1 - hp["ppo_clip"], 1 + hp["ppo_clip"]) * adv[j]
                loss = -torch.min(s1, s2).mean() \
                       + 0.5 * F.mse_loss(self.v(S[j]).squeeze(-1), ret[j])
                self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.reset_rollout()


Writing sixg/agents.py


In [ ]:
%%writefile sixg/runner.py
"""
Training / evaluation driver.

Methods
-------
  RR, WF, GRD      : classical non-learning heuristics
  SAC, TD3, PPO    : DRL baselines -- identical network capacity, optimiser,
                     batch size and number of environment interactions
  SAC+EHS          : ensemble seeding with best-of-seeds selection, no GWO
  SAC+GWO          : GWO refinement from a random initial population
  PROPOSED         : SAC + EHS + elitist GWO + behaviour cloning + adaptive gate
  PROPOSED-NG      : PROPOSED with the gate disabled (GWO at every slot)

Every method is evaluated on the SAME set of evaluation episode seeds, so the
channel realisations and packet arrivals are identical across methods.
"""

import time
import numpy as np

from .env import SixGEdgeEnv, project_action, inverse_project
from .heuristics import ANCHORS, ENSEMBLE, gwo_refine
from .agents import SAC, TD3, PPO, HP

HEURISTIC_METHODS = ("RR", "WF", "GRD", "URG", "PF")
ALG = dict(pop=12, iters=10, gate_margin=0.0)


def make_agent(method, sdim, adim, seed):
    if method == "TD3":
        return TD3(sdim, adim, seed=seed)
    if method == "PPO":
        return PPO(sdim, adim, seed=seed)
    if method.startswith("SAC") or method.startswith("PROPOSED"):
        return SAC(sdim, adim, seed=seed)
    return None


def decide(env, method, agent, state, rng, cfg, deterministic):
    """Return (b, p, rho), bc_target, timing."""
    if method in HEURISTIC_METHODS:
        t0 = time.perf_counter()
        act = ANCHORS[method](env)
        return act, None, dict(t_inf=time.perf_counter() - t0, t_ref=0.0, gwo=0)

    t0 = time.perf_counter()
    a_raw = agent.act(state, deterministic=deterministic)
    b, p, rho = project_action(a_raw, env.N)
    t_inf = time.perf_counter() - t0

    if method in ("SAC", "TD3", "PPO"):
        return (b, p, rho), None, dict(t_inf=t_inf, t_ref=0.0, gwo=0)

    use_ehs = ("EHS" in method) or method.startswith("PROPOSED")
    use_warm = method.startswith("PROPOSED+")
    f_sac = env.evaluate(b, p, rho)[0]
    t_ref = 0.0

    if method == "SAC+EHS":
        t0 = time.perf_counter()
        best, best_f = (b, p, rho), f_sac
        for key in ENSEMBLE:
            cand = ANCHORS[key](env)
            fc = env.evaluate(*cand)[0]
            if fc > best_f:
                best, best_f = cand, fc
        t_ref = time.perf_counter() - t0
        return best, inverse_project(*best), dict(t_inf=t_inf, t_ref=t_ref, gwo=0)

    # ---- GWO-refined variants ----
    t0 = time.perf_counter()
    if use_ehs:
        seeds = [(b, p, rho)] + [ANCHORS[k](env) for k in ENSEMBLE]
        # Temporal warm start: the channel is temporally correlated
        # (rho_fading = 0.9), so the previous slot's incumbent is a strong
        # seed.  This is the "solution carry-over" enhancement.
        if use_warm and cfg.get("_prev") is not None:
            seeds.append(cfg["_prev"])
    else:
        seeds = [(b, p, rho)]
        for _ in range(4):
            seeds.append((rng.dirichlet(np.ones(env.N)),
                          rng.dirichlet(np.ones(env.N)),
                          float(rng.uniform(0.2, 1.0))))
    t_ref += time.perf_counter() - t0

    if cfg.get("gate", False):
        t0 = time.perf_counter()
        f_anchor = max(env.evaluate(*s)[0] for s in seeds[1:])
        t_ref += time.perf_counter() - t0
        if f_sac >= f_anchor - cfg.get("gate_margin", 0.0):
            if use_warm:
                cfg["_prev"] = (b, p, rho)
            return (b, p, rho), None, dict(t_inf=t_inf, t_ref=t_ref, gwo=0)

    t0 = time.perf_counter()
    best, fbest, _ = gwo_refine(env, seeds, pop_size=cfg.get("pop", 10),
                                iters=cfg.get("iters", 10), rng=rng)
    t_ref += time.perf_counter() - t0
    if use_warm:
        cfg["_prev"] = best
    return best, inverse_project(*best), dict(t_inf=t_inf, t_ref=t_ref, gwo=1)


def run_one(method, n_users=30, seed=0, n_train_ep=120, n_eval_ep=15,
            cfg=None, log_curve=True):
    cfg = dict(ALG, **(cfg or {}))
    if method in ("PROPOSED", "PROPOSED+"):
        cfg.setdefault("gate", True)

    env = SixGEdgeEnv(n_users=n_users, seed=1000 + seed)
    rng = np.random.default_rng(2000 + seed)
    np.random.seed(3000 + seed)
    agent = make_agent(method, env.state_dim, env.action_dim, seed)
    curve = []

    # -------------------------------------------------------------- training
    if agent is not None:
        for ep in range(n_train_ep):
            s = env.reset(seed=10_000 + seed * 977 + ep)
            cfg["_prev"] = None
            ep_r = 0.0
            for _ in range(env.T):
                (b, p, rho), bc, _ = decide(env, method, agent, s, rng, cfg,
                                            deterministic=False)
                s2, r, done, _ = env.step(b, p, rho)
                a_store = inverse_project(b, p, rho)
                agent.store(s.astype(np.float32), a_store.astype(np.float32),
                            np.float32(r), s2.astype(np.float32), np.float32(done))
                agent.update(bc_target=bc)
                s = s2
                ep_r += r
            if log_curve:
                st = env.episode_stats()
                curve.append((ep_r / env.T, st["pdr"], st["goodput"]))

    # ------------------------------------------------------------ evaluation
    rows, t_infs, t_refs, gwos = [], [], [], []
    for k in range(n_eval_ep):
        s = env.reset(seed=777_000 + k)
        cfg["_prev"] = None
        thr_l, jain_l, ee_l, rho_l = [], [], [], []
        for _ in range(env.T):
            (b, p, rho), _, tm = decide(env, method, agent, s, rng, cfg,
                                        deterministic=True)
            t_infs.append(tm["t_inf"]); t_refs.append(tm["t_ref"]); gwos.append(tm["gwo"])
            s, r, done, info = env.step(b, p, rho)
            thr_l.append(float(info["thr"])); jain_l.append(float(info["jain"]))
            ee_l.append(float(info["ee"])); rho_l.append(rho)
        st = env.episode_stats()
        rows.append(dict(thr=np.mean(thr_l), jain=np.mean(jain_l), ee=np.mean(ee_l),
                         rho=np.mean(rho_l), pdr=st["pdr"], goodput=st["goodput"],
                         offered=st["offered"]))

    agg = {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}
    agg.update(method=method, seed=seed, n_users=n_users,
               t_inf_ms=float(np.mean(t_infs) * 1e3),
               t_ref_ms=float(np.mean(t_refs) * 1e3),
               gwo_rate=float(np.mean(gwos)))
    agg["t_tot_ms"] = agg["t_inf_ms"] + agg["t_ref_ms"]
    return agg, curve


Writing sixg/runner.py


## 4) سكربتات التجربة والتحليل والأشكال

In [ ]:
%%writefile run_experiment.py
"""
Unified, resumable experiment driver.

Run:
    python run_experiment.py

If interrupted (Ctrl+C, crash, power loss), simply run it again: completed
(exp, method, seed, n_users, w_pdr) combinations found in results.csv are
skipped automatically, so nothing is recomputed and nothing is duplicated.

Produces
--------
results/results.csv    -- one row per (experiment, method, seed) evaluation
results/curves.json    -- per-episode training curves (seed 0) for convergence plots
results/coldstart.json -- per-episode PDR during the first COLD_EP episodes, all seeds

Experiments
-----------
main   : N=30, 10 seeds (3 for the non-learning heuristics), all methods,
         used for the main comparison table + significance tests.
scale  : N in {50, 70, 90}, 5 seeds, a subset of methods, for scalability.
sens   : reliability weight w4 in {0.3, 0.6, 1.2}, 3 seeds, PROPOSED+ vs TD3.
cold   : first COLD_EP training episodes, 10 seeds, tests the paper's central
         cold-start claim directly (this is NOT the same as final performance).
"""
import csv, json, os, time
import numpy as np

from sixg.runner import run_one

OUT = "results"
RES = os.path.join(OUT, "results.csv")
CURVES = os.path.join(OUT, "curves.json")
COLD = os.path.join(OUT, "coldstart.json")

FIELDS = ["exp", "method", "seed", "n_users", "goodput", "offered", "pdr", "ee",
          "jain", "thr", "rho", "t_inf_ms", "t_ref_ms", "t_tot_ms", "gwo_rate",
          "w_pdr"]

TRAIN_EP, EVAL_EP = 120, 10
COLD_EP = 25

MAIN_METHODS = ["RR", "WF", "GRD", "PF", "PPO", "TD3", "SAC",
                "SAC+EHS", "SAC+GWO", "PROPOSED-NG", "PROPOSED", "PROPOSED+"]
MAIN_SEEDS = list(range(10))
# RR/WF/GRD/PF are deterministic (no learnable parameters, no stochastic
# policy) and are evaluated on the same fixed held-out episode set as every
# other method, so repeating them under different "seeds" reproduces bit-
# identical numbers -- there is no extra variance to sample.  One seed is
# both sufficient and correct; report their std as exactly 0 in the paper
# with this justification, not as a measurement artefact.
HEURISTIC_SEEDS = [0]

SCALE_N = [50, 70, 90]
SCALE_METHODS = ["RR", "PPO", "TD3", "SAC", "PROPOSED+"]
SCALE_SEEDS = list(range(5))

SENS_W = [0.3, 0.6, 1.2]
SENS_METHODS = ["PROPOSED+", "TD3"]
SENS_SEEDS = list(range(3))

COLD_METHODS = ["PPO", "TD3", "SAC", "SAC+GWO", "SAC+EHS", "PROPOSED+"]
COLD_SEEDS = list(range(10))


# ----------------------------------------------------------------------- I/O
def load_done():
    """Set of (exp, method, seed, n_users, w_pdr) already present in results.csv."""
    if not os.path.exists(RES):
        return set()
    import pandas as pd
    df = pd.read_csv(RES)
    keys = set()
    for _, r in df.iterrows():
        keys.add((r["exp"], r["method"], int(r["seed"]), int(r["n_users"]),
                  round(float(r["w_pdr"]), 3)))
    return keys


def writer():
    os.makedirs(OUT, exist_ok=True)
    new = not os.path.exists(RES)
    f = open(RES, "a", newline="")
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if new:
        w.writeheader()
    return f, w


def record(w, f, exp, agg, w_pdr=0.60):
    row = {k: agg.get(k, "") for k in FIELDS}
    row["exp"] = exp
    row["w_pdr"] = w_pdr
    w.writerow(row)
    f.flush()


def load_json(path):
    if os.path.exists(path):
        return json.load(open(path))
    return {}


def save_json(obj, path):
    json.dump(obj, open(path, "w"))


# --------------------------------------------------------------------- main
def main():
    t_start = time.time()
    f, w = writer()
    done = load_done()
    curves = load_json(CURVES)
    cold = load_json(COLD)
    print(f"resuming with {len(done)} completed runs already in {RES}", flush=True)

    def log(tag, extra, agg, t0):
        print(f"[{tag}] {extra:16s} pdr={agg['pdr']:6.2f} good={agg['goodput']:6.1f} "
              f"ee={agg['ee']:6.2f} jain={agg['jain']:.3f} tref={agg['t_ref_ms']:6.2f}ms "
              f"gwo={agg['gwo_rate']:.2f}  ({time.time()-t0:.0f}s, total "
              f"{time.time()-t_start:.0f}s)", flush=True)

    # ------------------------------------------------------------- 1. main
    for m in MAIN_METHODS:
        seeds = HEURISTIC_SEEDS if m in ("RR", "WF", "GRD", "PF") else MAIN_SEEDS
        for s in seeds:
            key = ("main", m, s, 30, 0.6)
            if key in done:
                continue
            t0 = time.time()
            agg, curve = run_one(m, n_users=30, seed=s,
                                 n_train_ep=TRAIN_EP, n_eval_ep=EVAL_EP)
            record(w, f, "main", agg)
            done.add(key)
            if s == 0 and curve:
                curves[m] = curve
                save_json(curves, CURVES)
            log("main", f"{m} s{s}", agg, t0)

    # ---------------------------------------------------------- 2. scaling
    for n in SCALE_N:
        for m in SCALE_METHODS:
            seeds = SCALE_SEEDS[:3] if m == "RR" else SCALE_SEEDS
            for s in seeds:
                key = ("scale", m, s, n, 0.6)
                if key in done:
                    continue
                t0 = time.time()
                agg, _ = run_one(m, n_users=n, seed=s, n_train_ep=TRAIN_EP,
                                 n_eval_ep=EVAL_EP, log_curve=False)
                record(w, f, "scale", agg)
                done.add(key)
                log(f"scale N={n}", f"{m} s{s}", agg, t0)

    # ------------------------------------------------------- 3. sensitivity
    import sixg.env as E
    base_w = E.CFG["W_PDR"]
    for wp in SENS_W:
        E.CFG["W_PDR"] = wp
        for m in SENS_METHODS:
            for s in SENS_SEEDS:
                key = ("sens", m, s, 30, round(wp, 3))
                if key in done:
                    continue
                t0 = time.time()
                agg, _ = run_one(m, n_users=30, seed=s, n_train_ep=TRAIN_EP,
                                 n_eval_ep=EVAL_EP, log_curve=False)
                record(w, f, "sens", agg, w_pdr=wp)
                done.add(key)
                log(f"sens w4={wp}", f"{m} s{s}", agg, t0)
    E.CFG["W_PDR"] = base_w

    # ------------------------------------------------------- 4. cold-start
    for m in COLD_METHODS:
        if m in cold and len(cold[m]) >= len(COLD_SEEDS):
            continue
        rows = cold.get(m, [])
        for s in range(len(rows), len(COLD_SEEDS)):
            t0 = time.time()
            _, curve = run_one(m, n_users=30, seed=s, n_train_ep=COLD_EP, n_eval_ep=1)
            rows.append([row[1] for row in curve])       # per-episode PDR
            cold[m] = rows
            save_json(cold, COLD)
            arr = np.array(rows)
            print(f"[cold] {m:12s} s{s}  ep1={arr[-1][0]:6.2f}  "
                  f"ep1-5_mean_so_far={arr[:, :5].mean():6.2f}  "
                  f"({time.time()-t0:.0f}s, total {time.time()-t_start:.0f}s)", flush=True)

    f.close()
    print(f"ALL EXPERIMENTS DONE in {time.time()-t_start:.0f}s", flush=True)


if __name__ == "__main__":
    main()


Writing run_experiment.py


In [ ]:
%%writefile analyze.py
"""Statistical analysis of the experiment campaign.

Reports mean +/- std, 95% confidence intervals (t-distribution), Welch's
unequal-variance t-test against the proposed method, Cliff's delta as a
non-parametric effect size, and Holm-Bonferroni correction across the family
of comparisons for each metric.
"""
import json
import os
import numpy as np
import pandas as pd
from scipy import stats

RES = "results/results.csv"
METRICS = ["pdr", "goodput", "ee", "jain", "t_tot_ms"]
LOWER_IS_BETTER = {"pdr", "t_tot_ms"}


def ci95(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return 0.0
    return stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))


def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))


def delta_label(d):
    d = abs(d)
    return ("negligible" if d < 0.147 else "small" if d < 0.33
            else "medium" if d < 0.474 else "large")


def holm(pvals):
    idx = np.argsort(pvals)
    adj = np.empty(len(pvals))
    running = 0.0
    for rank, i in enumerate(idx):
        val = (len(pvals) - rank) * pvals[i]
        running = max(running, val)
        adj[i] = min(running, 1.0)
    return adj


def summarise(df, group="method"):
    rows = []
    for m, g in df.groupby(group, sort=False):
        r = {"method": m, "n_seeds": len(g)}
        for k in METRICS:
            r[f"{k}_mean"] = g[k].mean()
            r[f"{k}_std"] = g[k].std(ddof=1) if len(g) > 1 else 0.0
            r[f"{k}_ci"] = ci95(g[k])
        rows.append(r)
    return pd.DataFrame(rows)


def compare(df, ref="PROPOSED+"):
    out = []
    for k in METRICS:
        a = df[df.method == ref][k].values
        raw = []
        others = [m for m in df.method.unique() if m != ref]
        for m in others:
            b = df[df.method == m][k].values
            if len(a) < 2 or len(b) < 2:
                raw.append((m, np.nan, np.nan)); continue
            t, p = stats.ttest_ind(a, b, equal_var=False)
            raw.append((m, p, cliffs_delta(a, b)))
        ps = np.array([r[1] for r in raw], dtype=float)
        mask = ~np.isnan(ps)
        adj = np.full(len(ps), np.nan)
        if mask.any():
            adj[mask] = holm(ps[mask])
        for (m, p, d), pa in zip(raw, adj):
            out.append(dict(metric=k, ref=ref, other=m, p_raw=p, p_holm=pa,
                            cliffs_delta=d, effect=delta_label(d) if d == d else "",
                            ref_mean=a.mean() if len(a) else np.nan,
                            other_mean=df[df.method == m][k].mean()))
    return pd.DataFrame(out)


if __name__ == "__main__":
    df = pd.read_csv(RES)
    main = df[df.exp == "main"]
    print("=" * 100)
    print("MAIN RESULTS (N=30), mean +/- std over seeds")
    print("=" * 100)
    s = summarise(main)
    disp = s[["method", "n_seeds"] + [f"{k}_{q}" for k in METRICS for q in ("mean", "std")]]
    print(disp.round(3).to_string(index=False))
    s.to_csv("results/summary_main.csv", index=False)

    if "PROPOSED+" in set(main.method):
        cmp = compare(main, "PROPOSED+")
        print("\nSIGNIFICANCE vs PROPOSED+ (Welch t-test, Holm-corrected)")
        print(cmp.round(4).to_string(index=False))
        cmp.to_csv("results/significance.csv", index=False)

    sc = df[df.exp == "scale"]
    if len(sc):
        print("\n" + "=" * 100)
        print("SCALABILITY (mean PDR %)")
        print(sc.pivot_table(index="n_users", columns="method", values="pdr").round(2).to_string())
        print("\nSCALABILITY (mean goodput Mbps)")
        print(sc.pivot_table(index="n_users", columns="method", values="goodput").round(1).to_string())

    se = df[df.exp == "sens"]
    if len(se):
        print("\n" + "=" * 100)
        print("SENSITIVITY to w4 (reliability weight)")
        print(se.pivot_table(index="w_pdr", columns="method",
                             values=["pdr", "ee"]).round(3).to_string())

    cold_path = "results/coldstart.json"
    if os.path.exists(cold_path):
        print("\n" + "=" * 100)
        print("COLD-START (mean PDR % over seeds, by training-episode window)")
        cold = json.load(open(cold_path))
        rows = []
        for m, seeds in cold.items():
            arr = np.array(seeds)                       # (n_seeds, n_episodes)
            rows.append(dict(
                method=m, n_seeds=arr.shape[0],
                ep1=arr[:, 0].mean(), ep1_ci=ci95(arr[:, 0]),
                ep1_5=arr[:, :5].mean(), ep1_5_ci=ci95(arr[:, :5].mean(axis=1)),
                ep1_25=arr.mean(), ep1_25_ci=ci95(arr.mean(axis=1)),
            ))
        cold_df = pd.DataFrame(rows)
        print(cold_df.round(3).to_string(index=False))
        cold_df.to_csv("results/coldstart_summary.csv", index=False)

        # significance of cold-start PDR: PROPOSED+ vs each baseline, ep1-5 window
        if "PROPOSED+" in cold:
            ref = np.array(cold["PROPOSED+"])[:, :5].mean(axis=1)
            raw = []
            for m, seeds in cold.items():
                if m == "PROPOSED+":
                    continue
                other = np.array(seeds)[:, :5].mean(axis=1)
                if len(ref) > 1 and len(other) > 1:
                    _, p = stats.ttest_ind(ref, other, equal_var=False)
                    d = cliffs_delta(ref, other)
                    raw.append((m, p, d))
            if raw:
                ps = np.array([r[1] for r in raw])
                adj = holm(ps)
                print("\nCOLD-START significance (episodes 1-5, PROPOSED+ vs baseline):")
                for (m, p, d), pa in zip(raw, adj):
                    print(f"  {m:12s} p_raw={p:.4f}  p_holm={pa:.4f}  "
                          f"cliffs_delta={d:+.3f} ({delta_label(d)})")



Writing analyze.py


In [ ]:
%%writefile make_figs.py
"""Publication-quality figures (300 dpi) from the campaign results."""
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RES = "results/results.csv"
CURVES = "results/curves.json"
OUT = "results/figs"
import os
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    "font.size": 11, "axes.labelsize": 12, "axes.titlesize": 12,
    "legend.fontsize": 10, "figure.dpi": 110, "savefig.dpi": 300,
    "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": 0.3,
})

STYLE = {
    "PROPOSED+":   dict(color="#12355b", lw=2.6, ls="-",  marker="o"),
    "PROPOSED":    dict(color="#1f77b4", lw=2.2, ls="-",  marker="s"),
    "PROPOSED-NG": dict(color="#5aa9e6", lw=1.8, ls="--", marker="^"),
    "SAC+GWO":     dict(color="#7f7f7f", lw=1.6, ls="-.", marker="v"),
    "SAC+EHS":     dict(color="#bcbd22", lw=1.6, ls="-.", marker="D"),
    "SAC":         dict(color="#d62728", lw=1.6, ls="--", marker="x"),
    "TD3":         dict(color="#ff7f0e", lw=1.6, ls="--", marker="+"),
    "PPO":         dict(color="#2ca02c", lw=1.6, ls="--", marker="*"),
    "RR":          dict(color="#8c564b", lw=1.4, ls=":",  marker="."),
    "WF":          dict(color="#9467bd", lw=1.4, ls=":",  marker="."),
    "GRD":         dict(color="#e377c2", lw=1.4, ls=":",  marker="."),
    "PF":          dict(color="#17becf", lw=1.4, ls=":",  marker="."),
}


def smooth(x, w=9):
    return pd.Series(x).rolling(w, min_periods=1).mean().values


def fig_convergence(curves, idx, ylabel, fname, title):
    plt.figure(figsize=(7, 4.4))
    for m, c in curves.items():
        if m not in STYLE:
            continue
        y = smooth([row[idx] for row in c])
        st = dict(STYLE[m]); st.pop("marker", None)
        plt.plot(y, label=m, **st)
    plt.xlabel("Training episode"); plt.ylabel(ylabel)
    plt.title(title); plt.legend(ncol=2)
    plt.savefig(f"{OUT}/{fname}"); plt.close()


def fig_bar_ci(df, metric, ylabel, fname, title, order):
    plt.figure(figsize=(7.4, 4.2))
    means, cis, labs = [], [], []
    from scipy import stats
    for m in order:
        g = df[df.method == m][metric].values
        if len(g) == 0:
            continue
        means.append(g.mean()); labs.append(m)
        cis.append(stats.t.ppf(0.975, len(g) - 1) * g.std(ddof=1) / np.sqrt(len(g))
                   if len(g) > 1 else 0.0)
    cols = [STYLE.get(m, {}).get("color", "#333") for m in labs]
    plt.bar(labs, means, yerr=cis, capsize=4, color=cols, alpha=0.85)
    plt.ylabel(ylabel); plt.title(title)
    plt.xticks(rotation=25, ha="right")
    plt.savefig(f"{OUT}/{fname}"); plt.close()


def fig_scale(df, metric, ylabel, fname, title):
    plt.figure(figsize=(7, 4.4))
    for m in df.method.unique():
        g = df[df.method == m].groupby("n_users")[metric]
        x = sorted(g.groups.keys())
        mu = [g.get_group(k).mean() for k in x]
        sd = [g.get_group(k).std(ddof=1) if len(g.get_group(k)) > 1 else 0 for k in x]
        st = dict(STYLE.get(m, dict(color="#333", lw=1.5, ls="-", marker="o")))
        plt.errorbar(x, mu, yerr=sd, capsize=3, label=m, **st)
    plt.xlabel("Number of users N"); plt.ylabel(ylabel)
    plt.title(title); plt.legend()
    plt.savefig(f"{OUT}/{fname}"); plt.close()


if __name__ == "__main__":
    df = pd.read_csv(RES)
    main = df[df.exp == "main"]
    order = [m for m in ["RR", "WF", "GRD", "PF", "PPO", "TD3", "SAC",
                         "SAC+EHS", "SAC+GWO", "PROPOSED-NG", "PROPOSED",
                         "PROPOSED+"] if m in set(main.method)]

    try:
        curves = json.load(open(CURVES))
        fig_convergence(curves, 1, "Packet drop rate (%)", "fig_conv_pdr.png",
                        "Convergence: deadline-violation rate")
        fig_convergence(curves, 2, "Goodput (Mbps)", "fig_conv_goodput.png",
                        "Convergence: delivered goodput")
        fig_convergence(curves, 0, "Average reward per slot", "fig_conv_reward.png",
                        "Convergence: objective F (Eq. 8)")
    except FileNotFoundError:
        pass

    fig_bar_ci(main, "pdr", "Packet drop rate (%)", "fig_pdr.png",
               "Reliability (mean, 95% CI)", order)
    fig_bar_ci(main, "goodput", "Goodput (Mbps)", "fig_goodput.png",
               "Delivered goodput (mean, 95% CI)", order)
    fig_bar_ci(main, "ee", "Energy efficiency (Mbit/J)", "fig_ee.png",
               "Energy efficiency (mean, 95% CI)", order)
    fig_bar_ci(main, "jain", "Jain index (satisfaction ratio)", "fig_jain.png",
               "Demand-aware fairness (mean, 95% CI)", order)

    sc = df[df.exp == "scale"]
    if len(sc):
        fig_scale(sc, "pdr", "Packet drop rate (%)", "fig_scale_pdr.png",
                  "Scalability: reliability vs user density")
        fig_scale(sc, "goodput", "Goodput (Mbps)", "fig_scale_goodput.png",
                  "Scalability: goodput vs user density")

    se = df[df.exp == "sens"]
    if len(se):
        plt.figure(figsize=(7, 4.2))
        for m in se.method.unique():
            g = se[se.method == m].groupby("w_pdr")
            x = sorted(g.groups.keys())
            plt.plot(x, [g.get_group(k)["pdr"].mean() for k in x],
                     label=f"{m} PDR", **{k: v for k, v in STYLE.get(m, {}).items()})
        plt.xlabel("Reliability weight $w_4$"); plt.ylabel("Packet drop rate (%)")
        plt.title("Sensitivity to the reliability weight"); plt.legend()
        plt.savefig(f"{OUT}/fig_sensitivity.png"); plt.close()

    cold_path = "results/coldstart.json"
    if os.path.exists(cold_path):
        cold = json.load(open(cold_path))
        plt.figure(figsize=(7, 4.4))
        for m, seeds in cold.items():
            if m not in STYLE:
                continue
            arr = np.array(seeds)
            mu = arr.mean(axis=0)
            sd = arr.std(axis=0, ddof=1) if arr.shape[0] > 1 else np.zeros_like(mu)
            x = np.arange(1, len(mu) + 1)
            st = dict(STYLE[m]); st.pop("marker", None)
            plt.plot(x, mu, label=m, **st)
            plt.fill_between(x, mu - sd, mu + sd, alpha=0.12,
                             color=STYLE[m]["color"])
        plt.xlabel("Training episode"); plt.ylabel("Packet drop rate (%)")
        plt.title("Cold-start reliability (mean $\\pm$ std over 10 seeds)")
        plt.legend(ncol=2)
        plt.savefig(f"{OUT}/fig_coldstart.png"); plt.close()

    print("figures written to", OUT)


Writing make_figs.py


## 5) استعادة النتائج الحقيقية الجزئية (اختياري لكن يُنصح به)

هذه 64 نتيجة **مقاسة فعلاً** من تشغيل سابق (RR/WF/GRD/PF، وPPO/TD3/SAC/SAC+EHS/SAC+GWO
كاملة بـ10 بذور، وPROPOSED ببذرتين). تنفيذ هذه الخلية يضعها في `results/` قبل
التشغيل، بحيث `run_experiment.py` يتجاهلها تلقائياً ولا يعيد حسابها — توفير حقيقي
للوقت. إن كنت تفضّل البدء من الصفر، تجاوز هذه الخلية فقط.


In [ ]:
%%writefile results/results.csv
exp,method,seed,n_users,goodput,offered,pdr,ee,jain,thr,rho,t_inf_ms,t_ref_ms,t_tot_ms,gwo_rate,w_pdr
main,RR,0,30,396.74,452.42473783528874,8.170590471459489,6.767413092988927,0.599510337659124,390.9728123398709,1.0,0.0061486975027946755,0.0,0.0061486975027946755,0.0,0.6
main,RR,1,30,396.74,452.42473783528874,8.170590471459489,6.767413092988927,0.599510337659124,390.9728123398709,1.0,0.006455210003650791,0.0,0.006455210003650791,0.0,0.6
main,RR,2,30,396.74,452.42473783528874,8.170590471459489,6.767413092988927,0.599510337659124,390.9728123398709,1.0,0.005555947502671188,0.0,0.005555947502671188,0.0,0.6
main,WF,0,30,385.25999999999993,452.42473783528874,10.675361136675884,6.590200465029288,0.5625164666600911,380.7347319118711,1.0,0.13562049750419192,0.0,0.13562049750419192,0.0,0.6
main,WF,1,30,385.25999999999993,452.42473783528874,10.675361136675884,6.590200465029288,0.5625164666600911,380.7347319118711,1.0,0.14183185249862618,0.0,0.14183185249862618,0.0,0.6
main,WF,2,30,385.25999999999993,452.42473783528874,10.675361136675884,6.590200465029288,0.5625164666600911,380.7347319118711,1.0,0.13385325500451017,0.0,0.13385325500451017,0.0,0.6
main,GRD,0,30,123.16000000000001,452.42473783528874,62.63568243053696,2.1256060806087733,0.19852459176708734,122.80234350158389,1.0,0.006634352502032925,0.0,0.006634352502032925,0.0,0.6
main,GRD,1,30,123.16000000000001,452.42473783528874,62.63568243053696,2.1256060806087733,0.19852459176708734,122.80234350158389,1.0,0.0065584650036498715,0.0,0.0065584650036498715,0.0,0.6
main,GRD,2,30,123.16000000000001,452.42473783528874,62.63568243053696,2.1256060806087733,0.19852459176708734,122.80234350158389,1.0,0.006315305004704896,0.0,0.006315305004704896,0.0,0.6
main,PF,0,30,342.26,452.42473783528874,18.540564810214427,5.874923352588391,0.5318820331867937,339.41112709999965,1.0,0.012415059999284495,0.0,0.012415059999284495,0.0,0.6
main,PF,1,30,342.26,452.42473783528874,18.540564810214427,5.874923352588391,0.5318820331867937,339.41112709999965,1.0,0.012853612497565337,0.0,0.012853612497565337,0.0,0.6
main,PF,2,30,342.26,452.42473783528874,18.540564810214427,5.874923352588391,0.5318820331867937,339.41112709999965,1.0,0.012589674995524547,0.0,0.012589674995524547,0.0,0.6
main,PPO,0,30,310.5,452.42473783528874,24.687917118191734,8.953173464913725,0.5204307863870381,306.2276669333304,0.5068346640565606,0.13918057250521088,0.0,0.13918057250521088,0.0,0.6
main,PPO,1,30,304.52000000000004,452.42473783528874,26.29820388276108,9.282426818485579,0.5118208495078111,299.9000390410553,0.46559178080391617,0.1357687025102905,0.0,0.1357687025102905,0.0,0.6
main,PPO,2,30,302.21999999999997,452.42473783528874,26.03680653325107,9.0631160748756,0.5115788844338021,298.5703780282316,0.4798957513663419,0.13225604749891318,0.0,0.13225604749891318,0.0,0.6
main,PPO,3,30,305.78000000000003,452.42473783528874,25.32957649766059,9.574855110010628,0.5149194275092043,301.372566607489,0.45150402360295655,0.13233489749723049,0.0,0.13233489749723049,0.0,0.6
main,PPO,4,30,305.23999999999995,452.42473783528874,26.110340221225908,9.825790204697205,0.5152182561739906,301.2469037097785,0.43286493125560954,0.1484192074974544,0.0,0.1484192074974544,0.0,0.6
main,PPO,5,30,315.96,452.42473783528874,23.48748489866346,8.927490800251933,0.5281095241233839,310.6806215962363,0.5197512480321677,0.14706936250433955,0.0,0.14706936250433955,0.0,0.6
main,PPO,6,30,303.1,452.42473783528874,26.18388411232104,8.350424250859806,0.5047207362202399,299.31845052750975,0.5400556833017212,0.1544375200057857,0.0,0.1544375200057857,0.0,0.6
main,PPO,7,30,306.72,452.42473783528874,25.304584415298944,9.101810827487757,0.5144886153811116,303.388071496846,0.4884506096261546,0.15365486499888448,0.0,0.15365486499888448,0.0,0.6
main,PPO,8,30,322.31999999999994,452.42473783528874,22.403753334937,9.36715532448869,0.5311762371629886,317.681778740632,0.5008448214985222,0.141289584996116,0.0,0.141289584996116,0.0,0.6
main,PPO,9,30,302.00000000000006,452.42473783528874,26.20419418407915,8.966381334241486,0.515304651472729,297.94649885038825,0.4878330377057063,0.13760264000097777,0.0,0.13760264000097777,0.0,0.6
main,TD3,0,30,379.74000000000007,452.42473783528874,11.521445841552074,9.58304838464507,0.5865484915955219,373.78184930063077,0.6867075178650672,0.12914812749670546,0.0,0.12914812749670546,0.0,0.6
main,TD3,1,30,320.28000000000003,452.42473783528874,23.073426373570747,17.751093518183527,0.5359309648913947,315.9475605643911,0.1632468542784242,0.12423160500645736,0.0,0.12423160500645736,0.0,0.6
main,TD3,2,30,321.86,452.42473783528874,22.716887910022088,17.823797373473315,0.5363863108714638,317.2383865032431,0.16324302490662476,0.14460317749978913,0.0,0.14460317749978913,0.0,0.6
main,TD3,3,30,322.65999999999997,452.42473783528874,22.574601274801097,17.9048788282633,0.539495226865557,318.68132157560177,0.16324278168785641,0.1379141125011074,0.0,0.1379141125011074,0.0,0.6
main,TD3,4,30,383.21999999999997,452.42473783528874,10.671778731360892,7.217343259523379,0.5877994325636452,377.9205449623349,0.8867550706267137,0.12071166499595165,0.0,0.12071166499595165,0.0,0.6
main,TD3,5,30,314.1,452.42473783528874,24.23297152889163,17.409011786930414,0.5221728281790798,309.8564544454817,0.16324389951950574,0.13342620501077818,0.0,0.13342620501077818,0.0,0.6
main,TD3,6,30,392.42,452.42473783528874,9.001407511505782,7.385625528427667,0.5971498683061343,386.7329861517384,0.8867572240789883,0.139602462501216,0.0,0.139602462501216,0.0,0.6
main,TD3,7,30,320.84,452.42473783528874,23.007189809148876,17.781786941936332,0.5325801567768378,316.514534306071,0.16327401739977193,0.1282925899977272,0.0,0.1282925899977272,0.0,0.6
main,TD3,8,30,321.65999999999997,452.42473783528874,22.70152498864999,17.80458817479025,0.5405468498153575,316.89649970888956,0.1632430543356737,0.13149664999559718,0.0,0.13149664999559718,0.0,0.6
main,TD3,9,30,322.96,452.42473783528874,22.570114117574946,17.88251849863827,0.5381535236937156,318.28335177424594,0.163242796818395,0.13638576250230017,0.0,0.13638576250230017,0.0,0.6
main,SAC,0,30,377.84000000000003,452.42473783528874,11.716460379273393,7.313238841876769,0.5877276226691951,372.88775128300676,0.8576124326062524,0.15862758250420939,0.0,0.15862758250420939,0.0,0.6
main,SAC,1,30,370.9,452.42473783528874,13.14062254814235,7.455851112043058,0.5721920689316559,366.1340759230905,0.8179984520232964,0.17051534500012622,0.0,0.17051534500012622,0.0,0.6
main,SAC,2,30,373.9,452.42473783528874,12.60188531829815,7.158570863182334,0.5775400047951349,369.1518538207864,0.8698523202715132,0.1679844275031428,0.0,0.1679844275031428,0.0,0.6
main,SAC,3,30,387.64,452.42473783528874,9.976153340290702,7.969785372989733,0.5926868243557462,381.6914150483849,0.7926027715635385,0.16437187750000248,0.0,0.16437187750000248,0.0,0.6
main,SAC,4,30,361.1,452.42473783528874,14.866202366579477,7.010911865775253,0.5676144931358971,355.7996982658405,0.8526897101868613,0.1631851524996364,0.0,0.1631851524996364,0.0,0.6
main,SAC,5,30,374.15999999999997,452.42473783528874,12.726447433732421,8.612673219004318,0.5803054352250474,368.41381911947497,0.6857880149076034,0.18040803750693613,0.0,0.18040803750693613,0.0,0.6
main,SAC,6,30,373.96,452.42473783528874,12.452339262549687,7.134955092864604,0.5830208652011238,368.8890081762114,0.872803246261425,0.15566590499815902,0.0,0.15566590499815902,0.0,0.6
main,SAC,7,30,388.94,452.42473783528874,9.62698764674382,7.5075265790530334,0.5976208620393728,382.61938847525244,0.857159258223105,0.19508445999747437,0.0,0.19508445999747437,0.0,0.6
main,SAC,8,30,375.73999999999995,452.42473783528874,12.371305659723953,7.263587219073893,0.5773857649176372,369.8217269968126,0.8561091245764274,0.15731476000155453,0.0,0.15731476000155453,0.0,0.6
main,SAC,9,30,373.41999999999996,452.42473783528874,12.664064768433375,7.148181542465747,0.5803449949350543,368.54979595554613,0.8697342865549105,0.19133221500680975,0.0,0.19133221500680975,0.0,0.6
main,SAC+EHS,0,30,394.46,452.42473783528874,8.331576234285757,9.59165329579713,0.5554858792009705,387.86108234730585,0.7760503729241022,0.17626548750399706,0.5069406849986535,0.6832061725026506,0.0,0.6
main,SAC+EHS,1,30,400.64,452.42473783528874,7.278867998875936,6.983809313717137,0.5770693435322937,393.66560563991874,0.969564987992505,0.1770756949895258,0.4900811649952174,0.6671568599847433,0.0,0.6
main,SAC+EHS,2,30,401.23999999999995,452.42473783528874,7.176137301958358,6.99270749083094,0.5782475933516628,394.4882773220747,0.9727411195340256,0.20109350000211634,0.5164627599924643,0.7175562599945806,0.0,0.6
main,SAC+EHS,3,30,402.08000000000004,452.42473783528874,7.032286987763366,7.093586144237309,0.5757691484672467,395.0027394895609,0.9587783012962767,0.18485650000684473,0.5026826975040422,0.6875391975108869,0.0,0.6
main,SAC+EHS,4,30,401.34000000000003,452.42473783528874,7.168446685540151,6.950727132333347,0.576913411945194,394.21851359271807,0.9769145039932253,0.18088441000031708,0.4939444774993263,0.6748288874996433,0.0,0.6
main,SAC+EHS,5,30,401.62,452.42473783528874,7.106558153040162,6.9009948453985785,0.577628025016658,394.75376710994533,0.9840420454052803,0.177191542500168,0.4881413949897251,0.6653329374898931,0.0,0.6
main,SAC+EHS,6,30,401.34,452.42473783528874,7.126920496276635,7.063941445413002,0.5773250749215106,393.91274137443617,0.9586987852981356,0.18099767998137395,0.49444874500863994,0.6754464249900138,0.0,0.6
main,SAC+EHS,7,30,401.78,452.42473783528874,7.067753693278773,6.941965120957891,0.5745557527011695,395.1895135803419,0.980443219874687,0.1668705925044378,0.4930210075053765,0.6598916000098143,0.0,0.6
main,SAC+EHS,8,30,401.84000000000003,452.42473783528874,7.09597777299017,6.9798158315711545,0.5774195997342271,394.97879697498126,0.9755343787520314,0.19087688998411068,0.5269269000064014,0.7178037899905121,0.0,0.6
main,SAC+EHS,9,30,401.32,452.42473783528874,7.16171543959813,6.862270693918598,0.5798775446781333,394.44230212627747,0.9905939505737121,0.2082513125208152,0.5402884224884019,0.7485397350092171,0.0,0.6
main,SAC+GWO,0,30,414.0,452.42473783528874,4.349936751359536,10.88129293252192,0.5507360413481044,401.52851080449085,0.6927803123623568,0.25293520750324205,2.411827527502055,2.664762735005297,1.0,0.6
main,SAC+GWO,1,30,414.2,452.42473783528874,4.313155565067123,10.803157319870733,0.5465886531081184,400.96147578312264,0.6992385351813204,0.24385308001569683,2.3761824274845367,2.6200355075002335,1.0,0.6
main,SAC+GWO,2,30,415.93999999999994,452.42473783528874,3.949835244861064,10.676589453660739,0.5524062207733571,403.408938845064,0.7089441415777492,0.2313055025047106,2.3115803200005303,2.542885822505241,1.0,0.6
main,SAC+GWO,3,30,412.32,452.42473783528874,4.571008647629932,10.97362808579619,0.5452812280493801,399.07428108081825,0.6888259944972728,0.22264222001467715,2.303917810005487,2.526560030020164,1.0,0.6
main,SAC+GWO,4,30,415.62000000000006,452.42473783528874,4.0433867126714285,10.480706403670853,0.549053437964786,401.6281764578788,0.7056443339697169,0.23382746499351015,2.3518531600052484,2.5856806249987585,1.0,0.6
main,SAC+GWO,5,30,411.43999999999994,452.42473783528874,4.854940719769456,10.964067136856196,0.5410760449320993,398.6786452213423,0.6827259468316724,0.24745822749196122,2.433027427524621,2.6804856550165823,1.0,0.6
main,SAC+GWO,6,30,414.9,452.42473783528874,4.227610078109614,10.850154260838327,0.5522306293925723,401.6777795177358,0.6888269471125523,0.22266172500962966,2.2613000849878517,2.4839618099974814,1.0,0.6
main,SAC+GWO,7,30,415.12,452.42473783528874,4.124662017136046,10.578099054663323,0.5526745203741871,402.4281543754823,0.7100877260545964,0.21778757250899616,2.2530462999998235,2.4708338725088197,1.0,0.6
main,SAC+GWO,8,30,413.52,452.42473783528874,4.440569903243447,10.605498354408903,0.5412851529900566,399.84381615202005,0.7009305126614069,0.22593144249754005,2.26356163499986,2.4894930774974,1.0,0.6
main,SAC+GWO,9,30,412.71999999999997,452.42473783528874,4.745054216972085,10.702968684043762,0.553523362558175,399.5623052248319,0.6963796428928107,0.22087981751838015,2.255509784985179,2.476389602503559,1.0,0.6
main,PROPOSED,0,30,410.6,452.42473783528874,5.001669303287511,9.821242493391901,0.5556745861083063,399.86158721376137,0.7879503982307169,0.20953560249949987,2.0332305174770227,2.2427661199765225,0.69,0.6
main,PROPOSED,1,30,404.65999999999997,452.42473783528874,6.2359719041698245,11.28222211874381,0.5399315055671831,393.63754125220964,0.6389753518664101,0.20779463000508258,1.964704815004552,2.1724994450096347,0.655,0.6


Writing results/results.csv


In [ ]:
%%writefile results/curves.json
{"PPO": [[0.24093699606635668, 18.039576622181315, 325.6], [0.21471879779669284, 13.780918727915195, 362.8], [0.2569039014727397, 17.87271142109852, 349.6], [0.23957363856108568, 12.911843276936777, 357.8], [0.3415920294414138, 12.286411716842961, 403.8], [0.386064824475869, 5.307262569832402, 389.4], [0.17929953610759028, 22.520852641334567, 303.0], [0.4080598729487529, 6.359649122807017, 402.0], [0.3178281233600564, 12.426284751474306, 391.4], [0.20036284348233097, 16.138698630136986, 360.2], [0.2537364963188198, 11.469690361971217, 381.2], [0.26341451803003246, 13.246519982038617, 354.2], [0.31810286564701584, 11.939615736505033, 361.2], [0.32818472799532894, 7.4563473336479476, 366.4], [0.35582105101787304, 11.683357176919408, 336.6], [0.3651059078549744, 10.708041958041958, 383.8], [0.25698364991090417, 11.758844045839561, 325.2], [0.27001681611609435, 14.149063371861299, 396.4], [0.23812563812561627, 17.998244073748904, 345.4], [0.40050712324942195, 9.168443496801705, 401.4], [0.3126156074288723, 13.516024152345564, 354.0], [0.19888846526361809, 19.45945945945946, 324.2], [0.1039241558764156, 29.707295762341634, 282.4], [0.22465476255919023, 20.032245062474807, 358.6], [0.3338314047888122, 10.357470210815766, 364.8], [0.2649494878162037, 12.010919017288444, 358.8], [0.20931872777314858, 15.871254162042176, 422.0], [0.32396298260314843, 13.224559896951481, 374.4], [0.31289277982869296, 13.148071339693074, 387.2], [0.2048866639979246, 15.746333045729077, 355.2], [0.2552922502505727, 14.834205933682373, 361.2], [0.1449844972236874, 26.49450023912004, 277.2], [0.2525826905985215, 16.645326504481435, 355.6], [0.2988560235270947, 14.314214463840399, 319.2], [0.35616247411379864, 9.129692832764505, 396.8], [0.09572534278095851, 25.269573370839193, 289.4], [0.1466687121509632, 22.424511545293072, 321.4], [0.31884906586951123, 13.440605773781353, 341.2], [0.29164444830288055, 16.17502458210423, 316.0], [0.382378777050869, 7.994323557237465, 364.4], [0.2356303611007286, 17.946428571428573, 339.6], [0.27597396553479375, 12.303547373147733, 366.4], [0.3252697735768748, 13.319755600814664, 387.8], [0.3003818071773365, 10.507726269315674, 378.0], [0.34493863527304425, 9.880068532267275, 295.0], [0.23898987436156077, 17.767747230201067, 364.0], [0.2870057422957424, 14.26344505066251, 407.8], [0.27696738401894444, 15.387894288150042, 369.2], [0.31147445164240845, 11.669741697416974, 355.0], [0.4069810746754329, 7.019655034095467, 429.0], [0.28938270439836455, 16.65979381443299, 372.8], [0.21884147831402911, 18.1311615945135, 349.0], [0.3153105974212266, 9.121338912133892, 413.0], [0.3206654042435134, 9.953051643192488, 358.8], [0.20855240223456714, 21.292281006071118, 333.0], [0.33514927555296364, 6.983849847228284, 400.8], [0.2602529367910792, 12.616593028964163, 331.6], [0.3031819067658815, 14.848883048620236, 353.0], [0.274033654956436, 14.854111405835544, 355.4], [0.35805535616465883, 7.674737322978529, 375.4], [0.29857898143428363, 13.943541488451668, 370.0], [0.2909929237030431, 10.206611570247935, 410.6], [0.1848489916105302, 20.816326530612244, 351.4], [0.33776536152292513, 10.354745925215724, 345.6], [0.3780277704332579, 10.091378625347636, 416.0], [0.22232213309929713, 13.116940862605603, 366.0], [0.15982846834579215, 17.31289449954914, 335.4], [0.24943271368004977, 14.252553389043639, 346.0], [0.3421050622329761, 11.544782251437962, 405.2], [0.33809579268970136, 10.218631178707223, 352.4], [0.16514299207762057, 18.89596602972399, 354.0], [0.2849832530079015, 15.017211703958692, 364.4], [0.2853893976508069, 15.043923865300146, 424.4], [0.33074483901409246, 13.053400273847558, 356.6], [0.23102567532467538, 18.48881384550443, 356.6], [0.29524805609099847, 14.341421143847487, 360.8], [0.23638358224307182, 17.204301075268816, 364.6], [0.42065920845059895, 6.472986748216106, 342.0], [0.35236532403625975, 10.905077262693156, 374.6], [0.2444527006459075, 14.342984409799554, 354.8], [0.1724173892944566, 19.4331983805668, 292.2], [0.3243217480915387, 11.266988163086365, 381.6], [0.29995835941843413, 14.577656675749319, 349.6], [0.21597026417272605, 19.88500239578342, 303.0], [0.28467928630082234, 17.454545454545453, 332.0], [0.27725161533769505, 11.19744058500914, 363.4], [0.2027702766666286, 19.83539094650206, 356.0], [0.2850796871972911, 14.393226716839134, 341.6], [0.22089733517513044, 17.579250720461093, 368.2], [0.2823915968222799, 13.601532567049809, 339.8], [0.3914436858176724, 8.763759525825572, 401.8], [0.3583918893645841, 9.187109187109186, 353.6], [0.2019209299919537, 19.10433539780848, 306.4], [0.3261716697193026, 12.11705532693187, 357.0], [0.2976662220604075, 11.294993793959454, 393.2], [0.32010520432313316, 15.840336134453782, 377.2], [0.3093224478841894, 14.959626009349765, 369.2], [0.3307194998936789, 11.594202898550725, 375.0], [0.16874158524547675, 22.28941684665227, 323.0], [0.2403351882021755, 9.387550200803213, 336.0], [0.28437736765307664, 12.533814247069431, 353.6], [0.25674421219481225, 11.871318531943816, 365.6], [0.21802131140988687, 20.218818380743983, 327.4], [0.3864843367824911, 10.453977509371095, 401.0], [0.09699301697945574, 25.26829268292683, 271.0], [0.3276728494815631, 7.735946364105209, 337.6], [0.22760192069072618, 17.899008115419296, 335.0], [0.34502706734826327, 9.67432950191571, 354.8], [0.20360089582125518, 16.9438202247191, 338.0], [0.33458734616953184, 9.797891036906854, 376.6], [0.24838169502353374, 19.728506787330318, 324.2], [0.10376503798099446, 30.314790858128504, 283.0], [0.2974645804048429, 13.301141352063214, 366.2], [0.2890707590132512, 11.597258829731155, 313.6], [0.24817989901615442, 20.097560975609756, 292.6], [0.2817048308989072, 14.867337602927721, 341.8], [0.3288183955820697, 10.544384502206965, 344.8], [0.27400621005294123, 15.111561866125761, 309.0], [0.18039103461930325, 26.452464788732396, 301.6], [0.36618842426730697, 11.12176414189837, 344.8]], "TD3": [[0.21174830269446035, 19.51219512195122, 319.6], [0.2308123984403841, 15.547703180212014, 353.4], [0.22812640118102173, 17.611159546643417, 348.2], [0.24613734194355824, 12.956366874443455, 362.6], [0.3488424785233707, 11.187957689178194, 411.6], [0.3984232826471016, 6.378026070763501, 384.0], [0.19237340086029855, 19.879518072289155, 316.4], [0.39586500207528513, 6.491228070175438, 403.2], [0.32131038678694257, 12.721145745577084, 387.6], [0.2619201855702738, 16.138698630136986, 359.2], [0.2449366701485542, 11.90580026166594, 377.0], [0.2353848840934832, 12.887292321508756, 352.2], [0.25613155553161765, 12.21408966148216, 357.4], [0.2650165121868335, 12.17555450684285, 346.8], [0.32283872449733575, 6.676204101096805, 364.6], [0.4761967869515152, 12.15034965034965, 374.4], [0.43943025920494244, 16.19332336821126, 305.2], [0.47557758074929657, 18.17457154244719, 377.4], [0.37630429504416635, 23.44161545215101, 319.6], [0.5320834693477885, 15.948827292110874, 368.8], [0.4354516421050887, 19.461216906641894, 326.6], [0.3475097130442645, 23.91891891891892, 304.0], [0.2045074741531984, 37.527304499781565, 246.0], [0.3138334903224582, 29.141475211608224, 312.6], [0.5175128112699062, 14.757103574702109, 345.4], [0.4855666575818504, 15.059144676979072, 346.4], [0.41165375999155795, 18.682944876063633, 401.4], [0.47161213755540815, 18.80635465865178, 345.2], [0.48184201494437956, 18.374118622978017, 361.2], [0.37445693631826754, 19.06816220880069, 342.6], [0.3860484189063108, 20.724258289703315, 333.0], [0.1953774252175597, 35.00717360114778, 243.0], [0.40258388539017104, 23.5595390524968, 323.6], [0.4130733758298642, 21.44638403990025, 291.4], [0.5411365015868304, 13.054607508532424, 375.0], [0.18415753450849146, 32.583216127519925, 256.4], [0.25184228754459353, 30.905861456483127, 279.4], [0.4211713729149381, 22.385234264079507, 303.0], [0.3840234425066562, 24.975417895771876, 276.8], [0.4866672820038632, 15.421002838221382, 336.2], [0.37202155339917203, 23.973214285714285, 312.6], [0.3923888488166043, 22.631342613381232, 315.0], [0.4733152257903268, 20.40733197556008, 351.4], [0.48100651192290844, 16.600441501103752, 351.2], [0.4238079288985519, 17.875499714448885, 265.0], [0.3452098474672944, 24.41526466967583, 335.4], [0.41835756803225876, 22.64224473889322, 363.6], [0.3596745382314167, 24.04092071611253, 323.4], [0.4379406861882097, 17.71217712177122, 328.4], [0.47416874170310086, 16.365824308062574, 376.0], [0.377335807236285, 25.030927835051546, 321.4], [0.31251717967432413, 24.38919845692242, 317.0], [0.5529715715316934, 11.92468619246862, 397.4], [0.5103839871128086, 15.258215962441314, 332.4], [0.3681443750512312, 27.233304423243712, 297.6], [0.5385396776680226, 10.912265386294195, 378.2], [0.4103755275997398, 20.569464899361808, 294.4], [0.38717663496988797, 26.149802890932982, 302.6], [0.45042678008156, 20.866489832007073, 326.8], [0.46725195089054894, 18.318867062585657, 325.6], [0.46233844023406495, 19.375534644995724, 343.0], [0.5100360568173588, 16.735537190082646, 375.8], [0.29314101511576085, 28.408163265306122, 312.4], [0.4433893758482152, 17.209971236816873, 313.4], [0.5561787055293805, 14.501390544298769, 390.8], [0.4266378351491949, 17.741218319253, 342.6], [0.3433256817920244, 22.54283137962128, 311.4], [0.3921068226003397, 16.38811513463324, 334.8], [0.516496213368782, 15.61216105176664, 380.8], [0.512400243185052, 14.020912547528518, 334.0], [0.31722474005810336, 24.628450106157114, 325.2], [0.42766059598886297, 21.04130808950086, 332.8], [0.4269326414626179, 21.888726207906295, 386.2], [0.4788503116787715, 18.575992697398448, 330.4], [0.3462306895573038, 27.142254115660617, 310.6], [0.43803605328254924, 19.150779896013866, 340.8], [0.4118715909785699, 21.83622828784119, 339.0], [0.51223953778245, 14.933741080530071, 307.4], [0.5107271534413352, 16.247240618101547, 350.6], [0.2947536601566855, 15.32293986636971, 352.4], [0.23255198820704495, 17.661943319838056, 303.2], [0.36377387464040833, 8.768084173608067, 395.2], [0.28592220830336923, 8.764759309718437, 380.0], [0.2356402944572719, 13.895543842836608, 332.6], [0.24495648491856264, 12.090909090909092, 361.4], [0.2960770825107298, 13.391224862888482, 353.4], [0.27339684610950543, 13.786008230452675, 390.8], [0.32679331473338513, 14.205079962370649, 344.8], [0.24151394366472262, 11.815561959654179, 399.4], [0.3708027747372315, 15.32567049808429, 330.8], [0.517225186445468, 7.578323454699407, 415.6], [0.38923611564690225, 3.415103415103415, 380.0], [0.2149648573090898, 13.19676036207718, 337.2], [0.5018147276900219, 11.88843164151806, 355.8], [0.3283026472008626, 7.90235829540753, 418.6], [0.4011924046328478, 9.705882352941176, 407.2], [0.3397113841515543, 10.922226944326392, 390.4], [0.4171134074794745, 18.093983311374615, 347.8], [0.1478823112585726, 17.840172786177106, 347.8], [0.20829614498369584, 11.44578313253012, 330.6], [0.33617709579828925, 4.824165915238954, 394.8], [0.21812729454850116, 7.838695061169008, 387.2], [0.1847431332218806, 13.435448577680525, 372.0], [0.3830842080341009, 3.7900874635568513, 440.6], [0.10217130147122959, 20.536585365853657, 296.6], [0.33041148559211864, 4.951005673027334, 354.6], [0.24282429713377188, 10.05410279531109, 377.2], [0.2817621841571868, 6.657088122605364, 372.6], [0.19676294919955167, 13.662921348314606, 358.6], [0.39340930664099966, 7.776801405975395, 391.8], [0.32025447492462955, 15.02262443438914, 348.8], [0.12838010708166633, 22.035360068995256, 324.8], [0.3631554130936311, 10.316066725197542, 386.8], [0.3934241880706494, 10.068529256721138, 323.2], [0.32732588589578404, 9.121951219512194, 346.0], [0.26986364974200205, 10.795974382433668, 364.2], [0.4870210671312979, 9.955860716037273, 350.4], [0.29040570055535153, 7.657200811359027, 344.6], [0.2712912011677645, 13.776408450704226, 362.8], [0.452902717710003, 7.861936720997123, 364.2]], "SAC": [[0.26498615808114956, 17.67142199723884, 329.8], [0.2691501065133063, 15.150176678445229, 355.4], [0.28492701970343715, 16.826503923278118, 353.6], [0.3174167798190813, 12.867319679430098, 360.0], [0.39846998357763636, 10.781122864117169, 412.4], [0.43513796109018177, 5.400372439478585, 386.8], [0.21921484023290186, 21.269694161260425, 311.6], [0.41793112868748983, 6.798245614035087, 399.8], [0.3648412326910971, 11.373209772535805, 392.0], [0.27995625262642465, 16.01027397260274, 360.8], [0.2908734655589364, 12.821631051024859, 375.8], [0.285509410127458, 10.821733273462057, 361.6], [0.3744324702335865, 9.240622140896615, 376.0], [0.3394824753801752, 7.881075979235488, 365.4], [0.40594265405268565, 9.537434430138292, 345.6], [0.35974252908635024, 8.653846153846153, 397.2], [0.3000440122901746, 13.004484304932735, 319.6], [0.3146811632857698, 11.518533280191312, 414.8], [0.27120991261042715, 16.944688323090432, 351.4], [0.4224901802265476, 9.381663113006397, 400.4], [0.3181041845056948, 12.772875058058522, 356.2], [0.20401295968544594, 18.46846846846847, 328.8], [0.1303811448977728, 27.959807776321536, 292.0], [0.24894179368311548, 19.911326078194275, 362.0], [0.38996548331778635, 9.99083409715857, 368.8], [0.3425550967214785, 13.193812556869881, 354.4], [0.22963033380239467, 17.055123936367, 414.0], [0.3517451022153036, 13.35337054529841, 373.0], [0.3840383785634386, 13.853172957279137, 383.6], [0.21582650253338137, 15.703192407247627, 362.0], [0.26055706592226896, 15.052356020942408, 359.6], [0.14894725456233066, 28.263988522238165, 272.2], [0.2561135016885855, 17.797695262483995, 353.6], [0.3271340559931123, 15.71072319201995, 315.6], [0.37693527691372186, 6.953924914675768, 406.2], [0.10961080508557688, 26.910454758556025, 280.8], [0.20043843350184595, 23.357015985790408, 314.6], [0.35755274871188897, 11.973497397065783, 348.4], [0.2935422479122082, 16.224188790560472, 313.0], [0.41224798291826453, 7.521286660359508, 369.0], [0.25782635868356907, 14.821428571428571, 352.2], [0.33144609811874676, 12.842388863942524, 364.0], [0.3358287801709624, 11.812627291242363, 402.8], [0.30810960888148337, 10.022075055187639, 383.4], [0.3351729631673066, 7.138777841233581, 305.6], [0.2637170136318537, 16.741895773491997, 372.8], [0.3452468681411194, 12.236944660950897, 420.6], [0.2772249258937559, 14.10912190963342, 374.0], [0.27242318878174454, 12.1309963099631, 355.0], [0.42787668571571, 6.578419574809467, 432.2], [0.2988419522337443, 13.31958762886598, 387.4], [0.2038614980569172, 18.08829832833262, 348.4], [0.2974796966466143, 7.489539748953975, 421.2], [0.3757491711072737, 7.84037558685446, 367.4], [0.25369066467392887, 15.091066782307026, 362.4], [0.3982916173621056, 4.670449585333915, 415.2], [0.2746931708829051, 11.536573392243495, 335.6], [0.2877425766213553, 14.542268944371441, 359.4], [0.3347314012705078, 12.37842617152962, 367.6], [0.3734436550839455, 5.4362722704431246, 389.0], [0.29444760488941524, 11.548331907613345, 386.4], [0.33056725985595714, 8.677685950413224, 417.6], [0.1688146681631374, 18.408163265306122, 363.4], [0.3406800221215641, 9.156279961649089, 355.6], [0.36671672482748735, 7.548669050456893, 433.4], [0.21858574411321277, 11.116051578479324, 376.8], [0.1687862918245019, 15.509467989179441, 345.6], [0.19350372693412826, 10.6313834726091, 361.8], [0.32614891670687063, 7.641741988496302, 424.4], [0.30962210871533, 7.652091254752852, 365.0], [0.20362672893348271, 16.008492569002122, 369.4], [0.26988914484181586, 13.42512908777969, 373.8], [0.2737957729416127, 12.554904831625183, 442.4], [0.3476537984345115, 9.310816978548608, 377.6], [0.26326233817752215, 16.167159138877164, 365.2], [0.27168947256995846, 11.178509532062392, 382.2], [0.2039554750256372, 11.827956989247312, 396.0], [0.39481788511559757, 4.638124362895005, 351.8], [0.3796651227319927, 7.637969094922737, 395.2], [0.23054572877116591, 12.249443207126948, 368.4], [0.14041642769860183, 14.827935222672064, 313.8], [0.32377487440759334, 9.557211749232792, 391.0], [0.3091174397509998, 11.262488646684831, 366.8], [0.22010483098993405, 15.764254911356014, 323.2], [0.2924013181499582, 13.181818181818182, 352.6], [0.31291543418116563, 7.1297989031078615, 382.8], [0.23580161683643105, 17.325102880658438, 372.8], [0.26162659582856523, 12.84101599247413, 349.8], [0.2519443610902256, 12.186084808563194, 394.6], [0.2557470758798157, 10.632183908045977, 355.0], [0.3932853105319325, 5.249788314987299, 423.6], [0.34807985817850706, 5.435305435305435, 369.0], [0.2052221912178113, 13.673177703668413, 330.0], [0.2901331584232179, 6.858710562414266, 383.4], [0.3327923224035971, 7.488622258998759, 419.0], [0.3570124945267517, 6.470588235294118, 425.2], [0.29377521462544254, 8.754781130471738, 403.4], [0.2865200266460378, 8.036890645586297, 400.0], [0.1579505466078219, 18.228941684665227, 343.0], [0.18801600967346044, 7.881526104417671, 347.4], [0.2664782560794164, 8.025247971145175, 378.4], [0.20159450085476216, 8.20117806977798, 384.4], [0.247574605485101, 14.704595185995624, 362.8], [0.39501446253601563, 5.372761349437734, 429.2], [0.10208385611485818, 22.048780487804876, 289.6], [0.264728131885414, 6.085611139762764, 349.0], [0.21429342214365868, 12.533814247069431, 364.2], [0.24494323998180495, 6.32183908045977, 374.0], [0.2100436998832878, 14.067415730337078, 355.0], [0.36089713467043494, 5.843585237258348, 402.6], [0.2964324953074521, 10.995475113122172, 369.8], [0.09657067876639638, 22.897800776196636, 321.2], [0.25826388831546815, 10.052677787532923, 386.4], [0.27158661611849577, 7.327358987875593, 331.4], [0.27768968310907666, 11.512195121951219, 332.2], [0.2646016551541395, 9.881061299176578, 369.0], [0.2800091501893522, 6.032368808239333, 368.2], [0.27113897654721225, 9.279918864097363, 336.2], [0.22449213186591996, 17.869718309859156, 343.0], [0.4019626710951708, 4.554170661553212, 376.2]], "SAC+EHS": [[0.3327962750844794, 5.7984353428439945, 386.8], [0.21918064282936972, 8.8339222614841, 390.8], [0.2955919631446138, 4.751525719267654, 418.0], [0.3204719776364554, 4.808548530721282, 405.8], [0.42838510931919976, 3.742880390561432, 455.8], [0.4804267444135558, 2.467411545623836, 404.4], [0.23321552534585424, 10.10194624652456, 365.6], [0.4989631211490638, 1.0964912280701755, 434.6], [0.3080127712994508, 5.307497893850043, 431.4], [0.33917929199117947, 7.277397260273973, 407.2], [0.28787705232050403, 6.279982555604012, 411.8], [0.3057491697940959, 4.714863044454423, 400.8], [0.40443472117805185, 4.117108874656908, 402.6], [0.34990455614583266, 3.303445021236432, 391.2], [0.5655815978471714, 0.619933237958989, 397.0], [0.423029528643026, 3.4965034965034967, 425.2], [0.27203111803371244, 5.4309915296462385, 359.4], [0.25585306837640603, 4.982064567556796, 454.2], [0.30238231404756993, 6.979806848112379, 399.8], [0.4342182680648361, 2.260127931769723, 443.0], [0.2592199520703468, 7.013469577333953, 386.2], [0.26094871238691736, 9.684684684684685, 375.8], [0.13259297040527548, 12.756662297946702, 371.6], [0.25163413228889503, 10.60056428859331, 417.8], [0.36502777124996744, 3.07057745187901, 404.8], [0.30873800899618886, 5.823475887170154, 395.0], [0.15705809299051118, 12.393636699963004, 446.4], [0.38289980975255744, 5.796479175611851, 417.4], [0.43524283007248893, 3.6914143508917463, 439.0], [0.1816117164852899, 10.82830025884383, 390.0], [0.28964226634847307, 7.635253054101222, 403.8], [0.19208150987610256, 14.299378287900526, 337.0], [0.2438979421487848, 8.408023900981647, 408.0], [0.41950508907376155, 5.187032418952619, 363.8], [0.39509332794721985, 2.8156996587030716, 437.6], [0.15076993680257128, 15.939990623534927, 331.2], [0.20873818583517068, 13.943161634103019, 365.8], [0.2396323654118863, 5.821107430194037, 382.6], [0.23880887379061083, 6.538839724680432, 361.4], [0.4492418736724309, 3.595080416272469, 392.2], [0.30596150918382703, 8.973214285714286, 382.2], [0.3302727893388321, 6.017063313875169, 403.8], [0.3919372751692144, 4.521384928716905, 446.4], [0.3458928302170382, 5.783664459161148, 408.2], [0.33826658847771957, 3.426613363792119, 321.6], [0.28805366172779406, 9.478867459991793, 419.6], [0.35890660644169314, 5.066250974279034, 466.4], [0.27273928947897746, 5.924978687127025, 420.2], [0.2846315462022229, 5.719557195571956, 391.2], [0.5538577711774586, 1.1632571199358204, 470.4], [0.4135203682664056, 4.041237113402062, 439.8], [0.2717415629478005, 11.101585940848693, 387.2], [0.28994439577987585, 4.7280334728033475, 436.8], [0.4071145751506937, 4.037558685446009, 389.2], [0.28244994405189605, 9.670424978317433, 395.0], [0.496717928337538, 1.8332605848974246, 431.0], [0.25714535417329737, 4.810996563573883, 369.6], [0.4241826774249734, 3.0661410424879545, 422.6], [0.389897259839251, 4.199823165340407, 412.2], [0.44778840970648587, 2.8323435358611238, 408.0], [0.2923658477110843, 5.260906757912746, 421.4], [0.3721564080517814, 5.578512396694215, 433.0], [0.18948980162336077, 14.0, 391.2], [0.48185242431152026, 1.9654841802492808, 392.6], [0.4337290702573699, 3.89352403655145, 460.0], [0.2715990478440486, 8.448199199644286, 393.2], [0.19975160643155881, 11.99278629395852, 363.0], [0.2623312289925205, 9.88857938718663, 368.6], [0.40942252672519786, 4.806902218570254, 444.8], [0.3112837459167149, 5.988593155893536, 377.6], [0.24906520116956612, 12.27176220806794, 388.8], [0.28379896545322125, 6.325301204819277, 411.8], [0.2883127503709849, 9.333821376281113, 464.6], [0.37745704946111813, 4.564125969876769, 402.0], [0.32046049487228984, 7.640354579991557, 413.4], [0.3817571259094254, 4.246100519930676, 419.0], [0.23392095069536434, 9.718775847808105, 409.8], [0.5536320093866863, 2.497451580020387, 363.6], [0.415458871686328, 3.7969094922737305, 411.6], [0.29006303588436144, 10.066815144766148, 379.2], [0.17727516987237882, 12.904858299595142, 323.8], [0.36797164823511175, 6.6637439719421305, 406.8], [0.3233366708124298, 6.630336058128973, 392.6], [0.2732928895886504, 10.20603737422137, 346.4], [0.3309726010789431, 8.272727272727273, 378.2], [0.38874130853779654, 4.43327239488117, 399.2], [0.3174919771765282, 9.91769547325103, 412.6], [0.3433513668623853, 9.0310442144873, 367.6], [0.32542958003066047, 9.180732811856732, 415.0], [0.3596275283754891, 5.363984674329502, 381.8], [0.4748914552005975, 3.4716342082980525, 437.8], [0.41897034718661247, 3.270803270803271, 380.6], [0.25207959473676783, 11.052882324916627, 346.2], [0.3776051790700049, 4.938271604938271, 395.6], [0.40370089073536725, 4.302854778651221, 439.4], [0.4649468723625615, 3.4453781512605044, 439.6], [0.34662989311887576, 6.4173395665108375, 415.0], [0.37015792070896636, 7.949055775142732, 399.6], [0.2172204366518918, 13.347732181425487, 372.4], [0.33712411672495496, 7.781124497991968, 347.2], [0.3811234450376166, 5.27502254283138, 395.2], [0.3387342787980332, 8.744902582691436, 381.4], [0.29189999006700734, 8.49015317286652, 391.2], [0.4742156125206233, 4.123281965847563, 438.8], [0.16174489147112067, 18.585365853658537, 305.8], [0.40110809853338686, 5.56988138215575, 352.2], [0.3001868578661996, 8.385933273219116, 386.4], [0.4159800311972237, 5.842911877394636, 376.4], [0.25490594268340533, 12.224719101123595, 362.6], [0.49046648171259155, 4.6572934973637965, 409.4], [0.3709473465352416, 6.108597285067873, 395.2], [0.17731183473447296, 14.445881845623113, 366.2], [0.30522463796254395, 7.6821773485513605, 397.6], [0.35114492966787497, 8.223510806536638, 328.6], [0.31698594232410227, 5.951219512195122, 362.2], [0.32356977826323907, 7.182067703568161, 380.8], [0.32522616058849213, 4.904364884747425, 368.6], [0.38454985685526166, 6.288032454361055, 350.2], [0.2958739352484913, 8.846830985915492, 389.8], [0.5279609163601513, 1.7737296260786193, 392.6]], "SAC+GWO": [[0.4688976026689472, 8.421537045559134, 374.2], [0.41033390749636195, 8.03886925795053, 393.2], [0.43660155050788624, 7.5850043591979075, 401.8], [0.5027583865143135, 6.055209260908281, 398.2], [0.6218414393217693, 3.2139951179820994, 456.6], [0.7986309246647781, 0.5121042830540037, 410.2], [0.3986440957570102, 10.611677479147358, 358.2], [0.7305861240844593, 1.2719298245614035, 431.2], [0.5223964122771587, 5.939342881213142, 421.2], [0.4980115203195744, 7.962328767123288, 402.8], [0.5087997087374083, 6.80331443523768, 405.8], [0.5087620872785491, 4.894476874719353, 397.8], [0.65281438067031, 4.254345837145471, 399.4], [0.5982643054127254, 3.445021236432279, 385.8], [0.7162384264975851, 0.7629947544110635, 389.8], [0.6413738028693062, 4.020979020979021, 424.4], [0.5525292667745025, 2.8400597907324365, 367.2], [0.5418907642447419, 5.7791948983658825, 449.4], [0.5352370194875092, 5.267778753292362, 411.4], [0.67658651074517, 2.08955223880597, 439.0], [0.5167892459174669, 10.125406409660938, 370.8], [0.455949395574597, 8.783783783783784, 375.4], [0.2099013375118472, 15.858453473132371, 352.8], [0.411024913176165, 10.07658202337767, 419.2], [0.6180225063742644, 3.1622364802933087, 403.4], [0.5745510523091296, 5.823475887170154, 392.4], [0.4078558847245035, 13.72549019607843, 435.2], [0.5995183188293746, 4.937741519965651, 416.8], [0.6454438902915905, 4.355039402737454, 433.2], [0.4055109164283845, 11.949956859361519, 379.8], [0.5143545045933687, 7.853403141361256, 402.0], [0.35614170990527994, 11.621233859397417, 347.0], [0.39229068062361294, 9.475032010243277, 396.4], [0.6153851986426069, 3.690773067331671, 370.0], [0.6430377724965685, 2.8156996587030716, 436.2], [0.2692166865484976, 17.674636661978433, 325.0], [0.33373624560074405, 13.676731793960924, 365.2], [0.4852679079824485, 6.625650733554188, 377.0], [0.4603172384478816, 6.735496558505408, 358.0], [0.7601469945950325, 2.034058656575213, 396.0], [0.5016537415652461, 8.125, 388.4], [0.5164256814466698, 5.523125280646609, 399.6], [0.5412380708129019, 3.380855397148676, 447.8], [0.5648389571659828, 5.518763796909492, 402.2], [0.5412682420324963, 2.8555111364934325, 325.2], [0.4346894379196512, 9.560935576528518, 415.8], [0.5562938317930595, 4.520654715510522, 463.8], [0.44758833317785474, 6.351236146632566, 413.0], [0.5049087741277181, 5.581180811808118, 389.2], [0.7351870777857992, 0.601684717208183, 468.8], [0.5754520034241015, 2.0618556701030926, 449.0], [0.43357302283453614, 9.901414487783969, 391.0], [0.5913185454228391, 4.97907949790795, 433.6], [0.6134377142590153, 4.553990610328638, 384.6], [0.44398587814423995, 7.372072853425846, 405.0], [0.6476904535249604, 1.9205587079877782, 428.0], [0.44979432485633775, 4.810996563573883, 368.0], [0.5592789923752938, 3.8545773105562855, 414.4], [0.5625943850524464, 4.951370468611848, 410.4], [0.650955601291466, 3.152124257651896, 401.6], [0.5230684979392553, 4.448246364414029, 420.8], [0.6037087875963552, 3.9669421487603307, 441.6], [0.33595381960738, 12.448979591836734, 395.6], [0.6503979848553239, 1.5819750719079577, 388.2], [0.6943166403697132, 1.946762018275725, 464.8], [0.4970701856184703, 8.670520231213873, 392.2], [0.35280104513258953, 10.504959422903516, 370.2], [0.434445186261127, 9.842154131847725, 367.0], [0.6461571728757216, 2.7937551355792936, 453.8], [0.5070046746866916, 5.180608365019012, 378.2], [0.4123501811232715, 10.488322717622081, 396.2], [0.45407327637521605, 6.024096385542169, 414.8], [0.4117794562081416, 6.698389458272328, 479.6], [0.5160923206015844, 3.4230944774075764, 406.2], [0.4194198705430736, 7.260447446179823, 412.4], [0.5551240826590864, 3.466204506065858, 421.2], [0.37566442379976495, 8.767576509511994, 416.4], [0.7472742952439074, 0.5606523955147809, 371.8], [0.6355521189979687, 1.1037527593818985, 426.8], [0.4202386796224945, 6.993318485523385, 395.4], [0.31192223428639426, 10.374493927125506, 336.0], [0.53959023237131, 2.9811486190267424, 422.6], [0.5630339633176045, 3.360581289736603, 405.4], [0.425054430872476, 6.229036895064686, 368.0], [0.5057088272874852, 4.318181818181818, 397.6], [0.5680837784555514, 2.742230347349177, 405.6], [0.4466698931225069, 5.88477366255144, 430.8], [0.5276456979457789, 4.374412041392286, 389.0], [0.4028825488347332, 5.351996706463566, 433.2], [0.5520822157977318, 2.346743295019157, 394.8], [0.693837761069618, 1.058425063505504, 449.0], [0.6574021166026421, 1.0101010101010102, 392.2], [0.41609484707792266, 5.145307289185326, 374.8], [0.5630329249169023, 2.652034750800183, 403.8], [0.5899589557963081, 1.5721969383533305, 452.6], [0.5999049550719653, 2.3529411764705883, 447.8], [0.5148918796013386, 3.867403314917127, 427.4], [0.598709755216272, 3.2059727711901624, 423.0], [0.3460862090127668, 9.416846652267818, 390.6], [0.5226342204067936, 3.1626506024096384, 367.4], [0.5499321405169607, 1.9837691614066726, 410.8], [0.5227487300528652, 3.896692342546443, 405.2], [0.4619446646827468, 3.588621444201313, 412.0], [0.634872291620172, 1.2078300708038316, 453.2], [0.27280776985883537, 12.682926829268293, 332.0], [0.6581275003043413, 2.0629190304280556, 366.8], [0.4509000119226931, 4.373309287646529, 404.8], [0.6453406548267852, 1.8199233716475096, 393.8], [0.3928749724454665, 7.50561797752809, 388.6], [0.6504702672550298, 1.9332161687170475, 424.4], [0.5083984255318588, 3.2126696832579187, 407.6], [0.2771941271845561, 12.332902112979733, 374.0], [0.42882435414845677, 4.8287971905179985, 412.4], [0.5355704872687692, 2.7411702688455457, 353.0], [0.4353389134275173, 3.1707317073170733, 376.4], [0.49365742838147764, 4.025617566331198, 396.2], [0.42384616263311825, 3.6292300147130945, 379.0], [0.6324222612781154, 1.3691683569979716, 370.4], [0.3967521017498347, 6.822183098591549, 399.8], [0.6192530867947197, 0.9587727708533078, 395.0]]}

Writing results/curves.json


## 6) تشغيل التجربة الكاملة

⚠️ **مهم لـ Colab**: هذه الخلية تعمل لعدة ساعات (2–4 ساعات تقريباً على CPU واحد).
- أبقِ تبويب المتصفح مفتوحاً (Colab المجاني يقطع الجلسة الخاملة بعد ~90 دقيقة بلا تفاعل).
- إن انقطعت الجلسة: أعد تشغيل كل الخلايا من الأعلى (Runtime → Run all). بما أن
  `results/` محفوظ في Drive، سيُكمل تلقائياً من حيث توقف ولن يُعيد أي شيء أُنجز فعلاً.
- لمراقبة التقدّم دون انتظار انتهاء الخلية، افتح خلية جديدة أثناء التشغيل ونفّذ:
  `!tail -20 /content/drive/MyDrive/heliyon_6g/results/results.csv`


In [ ]:
!python run_experiment.py


resuming with 64 completed runs already in results/results.csv
[main] PROPOSED-NG s0   pdr=  4.42 good= 413.2 ee= 11.38 jain=0.531 tref=  3.67ms gwo=1.00  (122s, total 123s)
[main] PROPOSED-NG s1   pdr=  4.58 good= 412.7 ee= 12.04 jain=0.538 tref=  5.53ms gwo=1.00  (102s, total 225s)
[main] PROPOSED-NG s2   pdr=  4.67 good= 411.4 ee= 12.02 jain=0.539 tref=  3.54ms gwo=1.00  (97s, total 323s)
[main] PROPOSED-NG s3   pdr=  4.60 good= 411.7 ee= 11.95 jain=0.536 tref=  5.69ms gwo=1.00  (97s, total 420s)
[main] PROPOSED-NG s4   pdr=  4.72 good= 411.3 ee= 11.81 jain=0.538 tref=  3.61ms gwo=1.00  (97s, total 518s)
[main] PROPOSED-NG s5   pdr=  4.33 good= 413.1 ee= 11.50 jain=0.543 tref=  3.54ms gwo=1.00  (96s, total 614s)
[main] PROPOSED-NG s6   pdr=  4.32 good= 414.0 ee= 11.29 jain=0.540 tref=  4.15ms gwo=1.00  (98s, total 711s)


## 7) التحليل الإحصائي

In [ ]:
!python analyze.py


## 8) توليد الأشكال (300 dpi)

In [ ]:
!python make_figs.py
import glob
from IPython.display import Image, display
for f in sorted(glob.glob("results/figs/*.png")):
    print(f)
    display(Image(f, width=650))


## 9) تنزيل النتائج كملف مضغوط

نتائج التشغيل محفوظة أصلاً بشكل دائم في Google Drive (`MyDrive/heliyon_6g/`).
هذه الخلية اختيارية فقط لتنزيل نسخة مضغوطة مباشرة إلى جهازك.


In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/heliyon_6g_results", "zip", PROJECT_DIR, "results")
files.download("/content/heliyon_6g_results.zip")
